# Cell 1: Import pandas and Path, load the Rossmann training CSV,
# and check whether the dataset was loaded successfully.

# Cell 2: Inspect the column names and data types to understand
# the structure of the dataset.

# Cell 3: Convert the Date column to datetime and identify the
# actual date range, number of unique dates, and number of stores.

# Cell 4: Count open and closed store-days, identify open days
# with zero sales, and check for missing sales values.

# Cell 5: Check missing values across all columns, summarize sales
# for open stores, and inspect date coverage for individual stores.

# Cell 6: Check gaps between consecutive observations for each store
# and inspect examples of open store-days with zero sales.

In [4]:

import pandas as pd
from pathlib import Path

# Change this path to the location of your Rossmann train.csv
DATA_PATH = Path("../data/train.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Place train.csv inside the project's data folder."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)


Dataset loaded successfully!
Shape: (1017209, 9)


C:\Users\KISHAN PANWAR\AppData\Local\Temp\ipykernel_27520\3860101610.py:13: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


In [5]:

print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)


Columns:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']

Data types:
Store             int64
DayOfWeek         int64
Date                str
Sales             int64
Customers         int64
Open              int64
Promo             int64
StateHoliday     object
SchoolHoliday     int64
dtype: object


In [6]:

df["Date"] = pd.to_datetime(df["Date"], errors="raise")

print("Earliest date:", df["Date"].min())
print("Latest date:", df["Date"].max())
print("Unique dates:", df["Date"].nunique())
print("Unique stores:", df["Store"].nunique())


Earliest date: 2013-01-01 00:00:00
Latest date: 2015-07-31 00:00:00
Unique dates: 942
Unique stores: 1115


In [7]:

print("Open/closed counts:")
print(df["Open"].value_counts(dropna=False).sort_index())

open_days = df[df["Open"] == 1].copy()

print("\nTotal open store-days:", len(open_days))
print("Open store-days with zero sales:",
      (open_days["Sales"] == 0).sum())

print("Open store-days with missing sales:",
      open_days["Sales"].isna().sum())

print("Closed store-days:", (df["Open"] == 0).sum())


Open/closed counts:
Open
0    172817
1    844392
Name: count, dtype: int64

Total open store-days: 844392
Open store-days with zero sales: 54
Open store-days with missing sales: 0
Closed store-days: 172817


In [8]:

print("Missing values per column:")
print(df.isna().sum())

print("\nSales summary for open stores:")
print(open_days["Sales"].describe())

print("\nStore coverage:")
print(df.groupby("Store")["Date"].agg(["min", "max", "count"]).head(10))


Missing values per column:
Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

Sales summary for open stores:
count    844392.000000
mean       6955.514291
std        3104.214680
min           0.000000
25%        4859.000000
50%        6369.000000
75%        8360.000000
max       41551.000000
Name: Sales, dtype: float64

Store coverage:
             min        max  count
Store                             
1     2013-01-01 2015-07-31    942
2     2013-01-01 2015-07-31    942
3     2013-01-01 2015-07-31    942
4     2013-01-01 2015-07-31    942
5     2013-01-01 2015-07-31    942
6     2013-01-01 2015-07-31    942
7     2013-01-01 2015-07-31    942
8     2013-01-01 2015-07-31    942
9     2013-01-01 2015-07-31    942
10    2013-01-01 2015-07-31    942


In [9]:

# Check whether each store has consecutive calendar dates
df_sorted = df.sort_values(["Store", "Date"]).copy()

df_sorted["days_since_previous_row"] = (
    df_sorted.groupby("Store")["Date"].diff().dt.days
)

print("Gaps between consecutive observations:")
print(
    df_sorted["days_since_previous_row"]
    .value_counts(dropna=False)
    .sort_index()
    .head(10)
)

# Inspect open days with zero sales
zero_sales = open_days[open_days["Sales"] == 0]

print("\nNumber of open zero-sales rows:", len(zero_sales))
print(zero_sales[["Store", "Date", "Sales", "Customers", "Promo"]].head(15))


Gaps between consecutive observations:
days_since_previous_row
1.0      1015914
185.0        180
NaN         1115
Name: count, dtype: int64

Number of open zero-sales rows: 54
        Store       Date  Sales  Customers  Promo
86825     971 2015-05-15      0          0      0
142278    674 2015-03-26      0          0      0
196938    699 2015-02-05      0          0      1
322053    708 2014-10-01      0          0      1
330176    357 2014-09-22      0          0      0
340348    227 2014-09-11      0          0      0
340860    835 2014-09-11      0          0      0
341795    835 2014-09-10      0          0      0
346232    548 2014-09-05      0          0      1
346734     28 2014-09-04      0          0      1
347669     28 2014-09-03      0          0      1
348604     28 2014-09-02      0          0      1
386065    102 2014-07-24      0          0      0
386173    238 2014-07-24      0          0      0
386227    303 2014-07-24      0          0      0


In [10]:
# Cell 7 — Find stores with 185-day gaps

# Identify stores with unusually long gaps between observations
long_gaps = df_sorted[
    df_sorted["days_since_previous_row"] > 1
].copy()

print("Number of long-gap intervals:", len(long_gaps))

print(
    long_gaps[
        ["Store", "Date", "days_since_previous_row"]
    ].sort_values("Store").to_string(index=False)
)


Number of long-gap intervals: 180
 Store       Date  days_since_previous_row
    13 2015-01-01                    185.0
    20 2015-01-01                    185.0
    22 2015-01-01                    185.0
    32 2015-01-01                    185.0
    36 2015-01-01                    185.0
    41 2015-01-01                    185.0
    46 2015-01-01                    185.0
    51 2015-01-01                    185.0
    52 2015-01-01                    185.0
    58 2015-01-01                    185.0
    72 2015-01-01                    185.0
    76 2015-01-01                    185.0
    81 2015-01-01                    185.0
    89 2015-01-01                    185.0
    99 2015-01-01                    185.0
   100 2015-01-01                    185.0
   108 2015-01-01                    185.0
   113 2015-01-01                    185.0
   115 2015-01-01                    185.0
   127 2015-01-01                    185.0
   129 2015-01-01                    185.0
   132 2015-01-01   

In [11]:
# Cell 8 — Investigate the 54 open zero-sales days

# Summarize zero-sales observations by store
zero_sales_by_store = (
    zero_sales.groupby("Store")
    .agg(
        zero_sales_days=("Sales", "size"),
        first_zero_date=("Date", "min"),
        last_zero_date=("Date", "max")
    )
    .sort_values("zero_sales_days", ascending=False)
)

print(zero_sales_by_store.to_string())

print("\nZero-sales days by weekday:")
print(zero_sales["DayOfWeek"].value_counts().sort_index())

print("\nZero-sales days by promo status:")
print(zero_sales["Promo"].value_counts(dropna=False))



       zero_sales_days first_zero_date last_zero_date
Store                                                
28                   3      2014-09-02     2014-09-04
25                   2      2014-02-12     2014-02-13
339                  2      2013-01-30     2013-01-31
1039                 2      2013-07-09     2013-07-10
1017                 2      2014-06-04     2014-06-05
983                  2      2014-01-17     2014-01-18
835                  2      2014-09-10     2014-09-11
665                  2      2013-06-28     2013-06-29
623                  2      2014-01-24     2014-01-25
364                  2      2013-05-07     2013-05-08
1100                 2      2014-04-29     2014-04-30
102                  2      2014-07-12     2014-07-24
353                  1      2013-03-16     2013-03-16
357                  1      2014-09-22     2014-09-22
57                   1      2014-07-01     2014-07-01
986                  1      2014-03-18     2014-03-18
227                  1      

In [12]:
# Cell 9 — Investigate StateHoliday

# Inspect distinct values and their Python types
print("Distinct StateHoliday values:")

for value in df["StateHoliday"].unique():
    print(repr(value), type(value).__name__)

print("\nStateHoliday value counts:")
print(df["StateHoliday"].value_counts(dropna=False))


Distinct StateHoliday values:
'0' str
'a' str
'b' str
'c' str
0 int

StateHoliday value counts:
StateHoliday
0    855087
0    131072
a     20260
b      6690
c      4100
Name: count, dtype: int64


In [13]:

# Inspect the dates immediately before and after each long gap
long_gaps["previous_date"] = (
    long_gaps["Date"]
    - pd.to_timedelta(long_gaps["days_since_previous_row"], unit="D")
)

print(
    long_gaps[
        ["Store", "previous_date", "Date", "days_since_previous_row"]
    ].head(10).to_string(index=False)
)

# Count stores with the 185-day gap
print("\nStores affected:", long_gaps["Store"].nunique())

# Check the overall number of rows during July–December 2014
missing_period = df[
    df["Date"].between("2014-07-01", "2014-12-31")
]

print("Rows in July–December 2014:", len(missing_period))
print("Stores represented in that period:", missing_period["Store"].nunique())


 Store previous_date       Date  days_since_previous_row
    13    2014-06-30 2015-01-01                    185.0
    20    2014-06-30 2015-01-01                    185.0
    22    2014-06-30 2015-01-01                    185.0
    32    2014-06-30 2015-01-01                    185.0
    36    2014-06-30 2015-01-01                    185.0
    41    2014-06-30 2015-01-01                    185.0
    46    2014-06-30 2015-01-01                    185.0
    51    2014-06-30 2015-01-01                    185.0
    52    2014-06-30 2015-01-01                    185.0
    58    2014-06-30 2015-01-01                    185.0

Stores affected: 180
Rows in July–December 2014: 172040
Stores represented in that period: 935


In [14]:
# Cell 10: Import Libraries and Load All Three CSV Files
import sqlite3
from pathlib import Path
import pandas as pd

# Locate the data folder relative to the notebooks directory
DATA_DIR = Path("../data")

train = pd.read_csv(DATA_DIR / "train.csv", low_memory=False)
store = pd.read_csv(DATA_DIR / "store.csv", low_memory=False)
test = pd.read_csv(DATA_DIR / "test.csv", low_memory=False)

# Convert dates to a consistent datetime format
train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])

# Normalize StateHoliday to strings for consistent category handling
train["StateHoliday"] = train["StateHoliday"].astype(str)
test["StateHoliday"] = test["StateHoliday"].astype(str)

print("Train:", train.shape)
print("Store:", store.shape)
print("Test:", test.shape)


Train: (1017209, 9)
Store: (1115, 10)
Test: (41088, 8)


In [15]:
# Cell 11: Create the SQLite Database and Load Tables

# Create the database file inside the data folder
DB_PATH = DATA_DIR / "rossmann.db"
conn = sqlite3.connect(DB_PATH)

# Load the DataFrames into SQLite tables
train.to_sql("train", conn, if_exists="replace", index=False)
store.to_sql("store", conn, if_exists="replace", index=False)
test.to_sql("test", conn, if_exists="replace", index=False)

print("SQLite database created:", DB_PATH.resolve())

# Verify that all three tables exist
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table'",
    conn
)

print(tables)


SQLite database created: C:\Users\KISHAN PANWAR\OneDrive\Documents\Desktop\DATASCIENCE\Projects\rossmann-demand-forecasting\data\rossmann.db
    name
0  train
1  store
2   test


In [16]:
# Cell 12: Calculate Store-Level Sales Aggregates Using SQL
query = """
SELECT
    Store,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN Open = 1 THEN 1 ELSE 0 END) AS open_days,
    SUM(CASE WHEN Open = 0 THEN 1 ELSE 0 END) AS closed_days,
    SUM(CASE WHEN Open = 1 THEN Sales ELSE 0 END) AS total_sales,
    AVG(CASE WHEN Open = 1 THEN Sales * 1.0 END) AS avg_open_day_sales,
    SUM(CASE WHEN Open = 1 AND Sales = 0 THEN 1 ELSE 0 END)
        AS open_zero_sales_days
FROM train
GROUP BY Store
ORDER BY Store;
"""

store_summary = pd.read_sql_query(query, conn)

print(store_summary.head(10))
print("\nNumber of stores aggregated:", len(store_summary))


   Store  total_rows  open_days  closed_days  total_sales  avg_open_day_sales  \
0      1         942        781          161      3716854         4759.096031   
1      2         942        784          158      3883858         4953.900510   
2      3         942        779          163      5408261         6942.568678   
3      4         942        784          158      7556507         9638.401786   
4      5         942        779          163      3642818         4676.274711   
5      6         942        780          162      4297758         5509.946154   
6      7         942        786          156      6930202         8817.050891   
7      8         942        784          158      4342857         5539.358418   
8      9         942        779          163      5112061         6562.337612   
9     10         942        784          158      4365642         5568.420918   

   open_zero_sales_days  
0                     0  
1                     0  
2                     0  
3   

In [17]:
# Cell 13: Calculate Overall Sales and Open/Closed Store-Day Counts
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT Store) AS unique_stores,
    COUNT(DISTINCT Date) AS unique_dates,
    SUM(CASE WHEN Open = 1 THEN 1 ELSE 0 END) AS open_store_days,
    SUM(CASE WHEN Open = 0 THEN 1 ELSE 0 END) AS closed_store_days,
    SUM(CASE WHEN Open = 1 AND Sales = 0 THEN 1 ELSE 0 END)
        AS open_zero_sales_days
FROM train;
"""

overall_summary = pd.read_sql_query(query, conn)
print(overall_summary)


   total_rows  unique_stores  unique_dates  open_store_days  \
0     1017209           1115           942           844392   

   closed_store_days  open_zero_sales_days  
0             172817                    54  


In [18]:
# Cell 14: Join Sales Data with Store Metadata Using SQL
query = """
SELECT
    s.Store,
    s.StoreType,
    s.Assortment,
    s.CompetitionDistance,
    COUNT(*) AS recorded_days,
    AVG(CASE WHEN t.Open = 1 THEN t.Sales * 1.0 END)
        AS avg_open_day_sales
FROM train AS t
JOIN store AS s
    ON t.Store = s.Store
GROUP BY
    s.Store,
    s.StoreType,
    s.Assortment,
    s.CompetitionDistance
ORDER BY avg_open_day_sales DESC
LIMIT 10;
"""

top_stores = pd.read_sql_query(query, conn)
print(top_stores)


   Store StoreType Assortment  CompetitionDistance  recorded_days  \
0    817         a          a                140.0            942   
1    262         b          a               1180.0            942   
2   1114         a          c                870.0            942   
3    251         a          c                340.0            942   
4    842         d          c               1200.0            758   
5    513         a          a                400.0            942   
6    562         b          c               1210.0            942   
7    788         a          c               1530.0            942   
8    383         a          c                350.0            942   
9    756         a          c                 50.0            942   

   avg_open_day_sales  
0        21757.483418  
1        20718.515924  
2        20666.562500  
3        19123.068036  
4        18574.795820  
5        18179.089286  
6        17969.556263  
7        17961.914541  
8        17294.716667  


In [19]:
# Cell 15: Validate SQL Results Against Pandas
# Independently calculate the number of open store-days in pandas
pandas_open_days = int((train["Open"] == 1).sum())

# Get the corresponding result from SQLite
sql_open_days = int(overall_summary.loc[0, "open_store_days"])

print("Pandas open store-days:", pandas_open_days)
print("SQL open store-days:", sql_open_days)

assert pandas_open_days == sql_open_days, "Open-day counts do not match!"

print("Validation passed!")


Pandas open store-days: 844392
SQL open store-days: 844392
Validation passed!


In [20]:

# Cell 16: Derive Calendar-Based Validation Folds and Final Holdout

HORIZON_DAYS = 42

data_start = train["Date"].min().normalize()
data_end = train["Date"].max().normalize()

# Reserve the final 42 calendar days as the untouched holdout
holdout_end = data_end
holdout_start = holdout_end - pd.Timedelta(days=HORIZON_DAYS - 1)

# Build four consecutive 42-day validation windows before the holdout
folds = []

for fold_number in range(4, 0, -1):
    fold_end = holdout_start - pd.Timedelta(
        days=(fold_number - 1) * HORIZON_DAYS + 1
    )
    fold_start = fold_end - pd.Timedelta(days=HORIZON_DAYS - 1)

    folds.append({
        "fold": 5 - fold_number,
        "train_start": data_start,
        "train_end": fold_start - pd.Timedelta(days=1),
        "validation_start": fold_start,
        "validation_end": fold_end,
    })

# Display folds from earliest to latest
fold_table = pd.DataFrame(folds)

print("Dataset range:", data_start.date(), "to", data_end.date())
print("\nValidation folds:")
print(fold_table.to_string(index=False))

print("\nFinal holdout:")
print(holdout_start.date(), "to", holdout_end.date())

# Verify that each window contains exactly 42 calendar days
for fold in folds:
    days = (fold["validation_end"] - fold["validation_start"]).days + 1
    assert days == HORIZON_DAYS

assert (holdout_end - holdout_start).days + 1 == HORIZON_DAYS

print("\nAll windows contain exactly 42 calendar days.")


Dataset range: 2013-01-01 to 2015-07-31

Validation folds:
 fold train_start  train_end validation_start validation_end
    1  2013-01-01 2015-01-02       2015-01-03     2015-02-13
    2  2013-01-01 2015-02-13       2015-02-14     2015-03-27
    3  2013-01-01 2015-03-27       2015-03-28     2015-05-08
    4  2013-01-01 2015-05-08       2015-05-09     2015-06-19

Final holdout:
2015-06-20 to 2015-07-31

All windows contain exactly 42 calendar days.


In [21]:

# Cell 17: Enforce the 42-Day Forecast-Origin Cutoff for Each Fold

for fold in folds:
    fold["train_end"] = (
        fold["validation_start"]
        - pd.Timedelta(days=HORIZON_DAYS)
    )

fold_table = pd.DataFrame(folds)

print("Leakage-aware validation folds:")
print(fold_table.to_string(index=False))

print("\nFinal holdout:")
print(holdout_start.date(), "to", holdout_end.date())

# Verify that training ends no later than 42 days before validation starts
for fold in folds:
    assert (
        fold["train_end"]
        <= fold["validation_start"] - pd.Timedelta(days=42)
    )

print("\nAll training cutoffs satisfy the 42-day rule.")


Leakage-aware validation folds:
 fold train_start  train_end validation_start validation_end
    1  2013-01-01 2014-11-22       2015-01-03     2015-02-13
    2  2013-01-01 2015-01-03       2015-02-14     2015-03-27
    3  2013-01-01 2015-02-14       2015-03-28     2015-05-08
    4  2013-01-01 2015-03-28       2015-05-09     2015-06-19

Final holdout:
2015-06-20 to 2015-07-31

All training cutoffs satisfy the 42-day rule.


In [22]:

# Cell 18: Define WAPE and Verify It on a Tiny Example

import numpy as np

def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = np.sum(np.abs(y_true))

    if denominator == 0:
        raise ValueError("WAPE is undefined when all actual values are zero.")

    return np.sum(np.abs(y_true - y_pred)) / denominator * 100


# Tiny example: actual sales and forecast sales
actual = [100, 200, 300]
forecast = [90, 220, 270]

print("WAPE:", round(wape(actual, forecast), 2), "%")


WAPE: 10.0 %


In [23]:

# Cell 19: Prepare Historical Sales Data for Baseline Forecasting

sales_data = train.copy()

sales_data["Date"] = pd.to_datetime(sales_data["Date"])
sales_data = sales_data.sort_values(["Store", "Date"]).reset_index(drop=True)

# Keep open store-days for the sales forecasting population
sales_data = sales_data[sales_data["Open"] == 1].copy()

# Normalize holiday categories consistently
sales_data["StateHoliday"] = sales_data["StateHoliday"].astype(str)

print("Open store-days:", len(sales_data))
print("Date range:", sales_data["Date"].min(), "to", sales_data["Date"].max())
print("Stores:", sales_data["Store"].nunique())


Open store-days: 844392
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00
Stores: 1115


In [24]:
# Cell 20: Build Efficient, Leakage-Safe Baseline Predictions

def predict_baselines(history, targets):
    history = history.loc[history["Open"] == 1].copy()
    targets = targets[["Store", "Date", "Sales"]].copy()

    history["Date"] = pd.to_datetime(history["Date"])
    targets["Date"] = pd.to_datetime(targets["Date"])

    # Map each target to the exact historical dates required by its lags
    result = targets.copy()
    result["lag42_date"] = result["Date"] - pd.Timedelta(days=42)
    result["lag364_date"] = result["Date"] - pd.Timedelta(days=364)

    lookup = history.set_index(["Store", "Date"])["Sales"]

    result["pred_lag42"] = pd.MultiIndex.from_frame(
        result[["Store", "lag42_date"]]
    ).map(lookup)

    result["pred_lag364"] = pd.MultiIndex.from_frame(
        result[["Store", "lag364_date"]]
    ).map(lookup)

    # Same-weekday mean: calculate from open-day sales within
    # the 56 calendar days ending at each target's t-42 cutoff.
    # Shift by 42 days so no target can use more recent sales.
    history = history.sort_values(["Store", "Date"])
    history["weekday"] = history["Date"].dt.dayofweek

    history["sales_shifted"] = history.groupby("Store")["Sales"].shift(42)

    # Use a date-indexed lookup for the shifted values
    shifted_lookup = history.set_index(["Store", "Date"])["sales_shifted"]

    # The lag baselines above are exact-date lookups. The rolling
    # baseline will be implemented separately after we validate these.
    result["pred_weekday_mean"] = np.nan

    return result.drop(columns=["lag42_date", "lag364_date"])

In [25]:
# Cell 21: Verify Fast Lag-Baseline Availability

first_fold = folds[0]

fold_targets = sales_data[
    sales_data["Date"].between(
        first_fold["validation_start"],
        first_fold["validation_end"]
    )
].copy()

baseline_predictions = predict_baselines(
    history=sales_data,
    targets=fold_targets
)

print("Total target rows:", len(baseline_predictions))

print("\nAvailable predictions:")
print(
    baseline_predictions[
        ["pred_lag42", "pred_lag364", "pred_weekday_mean"]
    ].notna().sum()
)

Total target rows: 39924

Available predictions:
pred_lag42           31042
pred_lag364          39529
pred_weekday_mean        0
dtype: int64


In [26]:
# Cell 22: Calculate Lag-42, Lag-364, and Eight-Week Weekday-Mean Baselines

def predict_baselines(history, targets):
    history = history.loc[history["Open"] == 1].copy()
    targets = targets[["Store", "Date", "Sales"]].copy()

    history["Date"] = pd.to_datetime(history["Date"])
    targets["Date"] = pd.to_datetime(targets["Date"])

    # Index historical open-day sales by store and date
    sales_lookup = history.set_index(["Store", "Date"])["Sales"]

    result = targets.copy()

    # Baseline 1: sales exactly 42 calendar days before target
    result["pred_lag42"] = np.nan
    lag42_keys = pd.MultiIndex.from_arrays([
        result["Store"],
        result["Date"] - pd.Timedelta(days=42)
    ])
    result["pred_lag42"] = lag42_keys.map(sales_lookup)

    # Baseline 2: sales exactly 364 calendar days before target
    lag364_keys = pd.MultiIndex.from_arrays([
        result["Store"],
        result["Date"] - pd.Timedelta(days=364)
    ])
    result["pred_lag364"] = lag364_keys.map(sales_lookup)

    # Baseline 3: mean of the eight previous same-weekday dates,
    # all dated t-42 or earlier: t-42, t-49, ..., t-91.
    lag_columns = []

    for lag in range(42, 92, 7):
        column = f"sales_lag_{lag}"
        keys = pd.MultiIndex.from_arrays([
            result["Store"],
            result["Date"] - pd.Timedelta(days=lag)
        ])
        result[column] = keys.map(sales_lookup)
        lag_columns.append(column)

    # Average available historical open-day sales.
    # Missing store-dates remain missing, never treated as zero.
    result["pred_weekday_mean"] = result[lag_columns].mean(axis=1)

    return result.drop(columns=lag_columns)

In [27]:
# Cell 23: Verify Baseline Availability and the t-42 Cutoff

first_fold = folds[0]

fold_targets = sales_data[
    sales_data["Date"].between(
        first_fold["validation_start"],
        first_fold["validation_end"]
    )
].copy()

baseline_predictions = predict_baselines(
    history=sales_data,
    targets=fold_targets
)

prediction_columns = [
    "pred_lag42",
    "pred_lag364",
    "pred_weekday_mean"
]

print("Total target rows:", len(baseline_predictions))
print("\nAvailable predictions:")
print(baseline_predictions[prediction_columns].notna().sum())

print("\nWeekday-mean examples:")
print(
    baseline_predictions[
        ["Store", "Date", "Sales", "pred_weekday_mean"]
    ].head(10)
)

assert (
    baseline_predictions["Date"] - pd.Timedelta(days=42)
    <= baseline_predictions["Date"]
).all()

print("\nBaseline availability check completed.")

Total target rows: 39924

Available predictions:
pred_lag42           31042
pred_lag364          39529
pred_weekday_mean    33799
dtype: int64

Weekday-mean examples:
     Store       Date  Sales  pred_weekday_mean
732      1 2015-01-03   5023           5116.375
734      1 2015-01-05   6239           4973.375
735      1 2015-01-06   4574           4574.875
736      1 2015-01-07   4796           4510.625
737      1 2015-01-08   3716           4544.125
738      1 2015-01-09   3998           4852.000
739      1 2015-01-10   4599           5330.250
741      1 2015-01-12   5346           5220.625
742      1 2015-01-13   4924           4703.375
743      1 2015-01-14   4541           4699.375

Baseline availability check completed.


In [28]:

# Cell 24: Evaluate Baselines on Common Eligible Targets Across Four Folds

baseline_columns = [
    "pred_lag42",
    "pred_lag364",
    "pred_weekday_mean"
]

fold_results = []
fold_predictions = []

for fold in folds:
    targets = sales_data[
        sales_data["Date"].between(
            fold["validation_start"],
            fold["validation_end"]
        )
    ].copy()

    predictions = predict_baselines(
        history=sales_data,
        targets=targets
    )

    # Use the intersection of eligible targets for a fair comparison
    common = predictions.dropna(subset=baseline_columns).copy()

    print(
        f"Fold {fold['fold']}: {len(targets):,} open targets; "
        f"{len(common):,} common eligible targets"
    )

    actual = common["Sales"].to_numpy(dtype=float)
    total_actual = actual.sum()

    if total_actual <= 0:
        raise ValueError("WAPE requires positive total actual sales.")

    for column in baseline_columns:
        forecast = common[column].to_numpy(dtype=float)

        error_wape = wape(actual, forecast)
        bias_ratio = forecast.sum() / total_actual

        fold_results.append({
            "fold": fold["fold"],
            "baseline": column,
            "eligible_targets": len(common),
            "WAPE_pct": error_wape,
            "forecast_to_actual_ratio": bias_ratio
        })

    common["fold"] = fold["fold"]
    fold_predictions.append(common)

baseline_results = pd.DataFrame(fold_results)

print("\nBaseline results:")
print(
    baseline_results.round(4).to_string(index=False)
)


Fold 1: 39,924 open targets; 30,832 common eligible targets
Fold 2: 40,176 open targets; 39,703 common eligible targets
Fold 3: 37,044 open targets; 33,600 common eligible targets
Fold 4: 37,473 open targets; 31,279 common eligible targets

Baseline results:
 fold          baseline  eligible_targets  WAPE_pct  forecast_to_actual_ratio
    1        pred_lag42             30832   28.5905                    1.2302
    1       pred_lag364             30832   20.3782                    0.9674
    1 pred_weekday_mean             30832   17.0310                    1.1016
    2        pred_lag42             39703   19.8226                    1.0220
    2       pred_lag364             39703   10.6102                    0.9682
    2 pred_weekday_mean             39703   18.6158                    1.1323
    3        pred_lag42             33600   18.2166                    0.8785
    3       pred_lag364             33600   15.3309                    0.9757
    3 pred_weekday_mean             336

In [29]:

# Cell 25: Summarize Mean WAPE, Fold Range, Bias, and Relative Error

summary = (
    baseline_results.groupby("baseline")
    .agg(
        mean_WAPE_pct=("WAPE_pct", "mean"),
        min_WAPE_pct=("WAPE_pct", "min"),
        max_WAPE_pct=("WAPE_pct", "max"),
        mean_forecast_to_actual_ratio=(
            "forecast_to_actual_ratio", "mean"
        )
    )
    .reset_index()
)

best_baseline_wape = summary["mean_WAPE_pct"].min()

summary["scaled_error_vs_best"] = (
    summary["mean_WAPE_pct"] / best_baseline_wape
)

print(summary.round(4).to_string(index=False))


         baseline  mean_WAPE_pct  min_WAPE_pct  max_WAPE_pct  mean_forecast_to_actual_ratio  scaled_error_vs_best
      pred_lag364        14.6547       10.6102       20.3782                         0.9708                1.0000
       pred_lag42        23.5622       18.2166       28.5905                         1.0388                1.6078
pred_weekday_mean        17.2650       14.0615       19.3515                         1.0198                1.1781


In [30]:
# Cell 26: Build leakage-aware forecasting features

import numpy as np
# Work only with open store-days as forecasting targets.
feature_data = sales_data.loc[sales_data["Open"] == 1].copy()
feature_data["Date"] = pd.to_datetime(feature_data["Date"])
feature_data["StateHoliday"] = feature_data["StateHoliday"].astype(str)
feature_data = feature_data.sort_values(["Store", "Date"]).reset_index(drop=True)

# Sales lookup: (store, date) -> observed sales on an open day.
sales_lookup = (
    feature_data.set_index(["Store", "Date"])["Sales"]
    .sort_index()
)

features = feature_data[
    ["Store", "Date", "Sales", "Promo", "StateHoliday", "SchoolHoliday"]
].copy()

# Calendar features known for the target date.
features["DayOfWeek"] = features["Date"].dt.dayofweek + 1
features["Month"] = features["Date"].dt.month
features["DayOfMonth"] = features["Date"].dt.day
features["WeekOfYear"] = features["Date"].dt.isocalendar().week.astype(int)
features["IsWeekend"] = (features["Date"].dt.dayofweek >= 5).astype(int)

# Exact sales lags. Every source date is on or before target_date - 42 days.
for lag in [42, 49, 364]:
    source_dates = features["Date"] - pd.to_timedelta(lag, unit="D")
    keys = pd.MultiIndex.from_arrays(
        [features["Store"], source_dates],
        names=["Store", "Date"]
    )
    features[f"Sales_lag_{lag}"] = keys.map(sales_lookup)

# Mean of the eight previous same-weekday observations:
# target_date - 42, -49, ..., -91 days.
same_weekday_lags = []
for lag in range(42, 92, 7):
    source_dates = features["Date"] - pd.to_timedelta(lag, unit="D")
    keys = pd.MultiIndex.from_arrays(
        [features["Store"], source_dates],
        names=["Store", "Date"]
    )
    column = f"_weekday_sales_{lag}"
    features[column] = keys.map(sales_lookup)
    same_weekday_lags.append(column)

features["Sales_same_weekday_mean_8w"] = features[
    same_weekday_lags
].mean(axis=1)

features = features.drop(columns=same_weekday_lags)

# Rolling mean over the 56 calendar days ending at each source date.
# Shift the resulting feature forward by 42 days so it belongs to target_date.
# Only open-day sales enter this mean; missing/closed dates are not treated as zero.
rolling_parts = []

for store_id, group in feature_data.groupby("Store", sort=False):
    group = group.sort_values("Date").copy()
    group = group.set_index("Date")

    rolling_mean = group["Sales"].rolling(
        "56D",
        closed="right",
        min_periods=1
    ).mean()

    part = pd.DataFrame({
        "Store": store_id,
        "Date": rolling_mean.index + pd.Timedelta(days=42),
        "Sales_rolling_mean_56d": rolling_mean.to_numpy()
    })
    rolling_parts.append(part)

rolling_features = pd.concat(rolling_parts, ignore_index=True)
rolling_lookup = rolling_features.set_index(["Store", "Date"])[
    "Sales_rolling_mean_56d"
]

rolling_keys = pd.MultiIndex.from_arrays(
    [features["Store"], features["Date"]],
    names=["Store", "Date"]
)
features["Sales_rolling_mean_56d"] = rolling_keys.map(rolling_lookup)

# Add static store information.
store_metadata = pd.read_csv(DATA_DIR / "store.csv")
features = features.merge(
    store_metadata,
    on="Store",
    how="left",
    validate="many_to_one"
)

# One-hot encode categorical metadata and holiday labels.
categorical_columns = [
    column for column in [
        "StateHoliday", "StoreType", "Assortment", "PromoInterval"
    ]
    if column in features.columns
]

features = pd.get_dummies(
    features,
    columns=categorical_columns,
    dummy_na=True,
    dtype=int
)

# Store the final feature table in memory; no files are overwritten.
features = features.sort_values(["Date", "Store"]).reset_index(drop=True)

print("Feature table shape:", features.shape)
print("Target rows:", len(features))
print("Date range:", features["Date"].min().date(), "to",
      features["Date"].max().date())
print("\nMissing values in sales-derived features:")
print(
    features[
        [
            "Sales_lag_42",
            "Sales_lag_49",
            "Sales_lag_364",
            "Sales_same_weekday_mean_8w",
            "Sales_rolling_mean_56d"
        ]
    ].isna().sum()
)

KeyboardInterrupt: 

In [ ]:
# Cell 27: Verify engineered sales-feature cutoff dates

feature_cutoff = features["Date"] - pd.Timedelta(days=42)

assert (features["Date"] - pd.Timedelta(days=42) <= features["Date"]).all()

# The exact lag columns must point to the expected source dates.
for lag in [42, 49, 364]:
    expected_source_date = features["Date"] - pd.to_timedelta(lag, unit="D")
    assert (expected_source_date <= feature_cutoff).all()

# The eight-week same-weekday mean uses dates from t-42 through t-91.
assert ((features["Date"] - pd.Timedelta(days=42)) <= feature_cutoff).all()
assert ((features["Date"] - pd.Timedelta(days=91)) <= feature_cutoff).all()

print("Basic date-boundary checks passed.")
print("Important: this checks date logic, not yet the full feature provenance.")

Basic date-boundary checks passed.
Important: this checks date logic, not yet the full feature provenance.


In [ ]:
# Cell 28: Test feature provenance against historical sales

rng = np.random.default_rng(42)
sample_indices = rng.choice(
    features.index.to_numpy(),
    size=min(300, len(features)),
    replace=False
)
sample = features.loc[sample_indices]

history_by_store = {
    store_id: group.sort_values("Date")
    for store_id, group in feature_data.groupby("Store")
}

def expected_sales(store_id, source_date):
    # Return sales only when that store has an observed open-day record.
    key = (store_id, pd.Timestamp(source_date))
    return sales_lookup.get(key, np.nan)

for _, row in sample.iterrows():
    store_id = row["Store"]
    target_date = pd.Timestamp(row["Date"])
    cutoff = target_date - pd.Timedelta(days=42)

    # Verify exact lags.
    for lag in [42, 49, 364]:
        expected = expected_sales(
            store_id,
            target_date - pd.Timedelta(days=lag)
        )
        actual = row[f"Sales_lag_{lag}"]

        assert (
            (pd.isna(expected) and pd.isna(actual))
            or (
                pd.notna(expected)
                and pd.notna(actual)
                and np.isclose(expected, actual)
            )
        ), f"Lag {lag} mismatch: Store {store_id}, target {target_date.date()}"

    # Verify the same-weekday mean from the eight exact historical dates.
    weekday_values = [
        expected_sales(
            store_id,
            target_date - pd.Timedelta(days=lag)
        )
        for lag in range(42, 92, 7)
    ]
    expected_weekday_mean = (
        np.nanmean(weekday_values)
        if not np.isnan(weekday_values).all()
        else np.nan
    )
    actual_weekday_mean = row["Sales_same_weekday_mean_8w"]

    assert (
        (pd.isna(expected_weekday_mean) and pd.isna(actual_weekday_mean))
        or (
            pd.notna(expected_weekday_mean)
            and pd.notna(actual_weekday_mean)
            and np.isclose(expected_weekday_mean, actual_weekday_mean)
        )
    ), f"Same-weekday mean mismatch: Store {store_id}, target {target_date.date()}"

    # Verify the rolling mean over (cutoff - 56 days, cutoff].
    store_history = history_by_store[store_id]
    window_start = cutoff - pd.Timedelta(days=56)

    historical_window = store_history.loc[
        (store_history["Date"] > window_start)
        & (store_history["Date"] <= cutoff),
        "Sales"
    ]

    expected_rolling_mean = (
        historical_window.mean()
        if len(historical_window) > 0
        else np.nan
    )
    actual_rolling_mean = row["Sales_rolling_mean_56d"]

    assert (
        (pd.isna(expected_rolling_mean) and pd.isna(actual_rolling_mean))
        or (
            pd.notna(expected_rolling_mean)
            and pd.notna(actual_rolling_mean)
            and np.isclose(expected_rolling_mean, actual_rolling_mean)
        )
    ), f"Rolling mean mismatch: Store {store_id}, target {target_date.date()}"

print(f"All provenance checks passed for {len(sample)} sampled rows.")

AssertionError: Rolling mean mismatch: Store 4, target 2014-02-06

In [ ]:
# Cell 29: Define fold origins and verify training-label availability

folds_checked = []

for fold in folds:
    validation_start = pd.Timestamp(fold["validation_start"])
    validation_end = pd.Timestamp(fold["validation_end"])
    forecast_origin = validation_start - pd.Timedelta(days=1)

    train_mask = features["Date"] <= forecast_origin
    validation_mask = features["Date"].between(
        validation_start,
        validation_end
    )

    train_dates = features.loc[train_mask, "Date"]
    validation_dates = features.loc[validation_mask, "Date"]

    assert len(validation_dates) > 0
    assert train_dates.max() <= forecast_origin
    assert validation_dates.min() == validation_start
    assert validation_dates.max() == validation_end
    assert (validation_end - validation_start).days == 41

    folds_checked.append({
        "fold": fold["fold"],
        "forecast_origin": forecast_origin.date(),
        "train_last_target_date": train_dates.max().date(),
        "validation_start": validation_start.date(),
        "validation_end": validation_end.date(),
        "training_rows": int(train_mask.sum()),
        "validation_open_store_days": int(validation_mask.sum())
    })

fold_check_table = pd.DataFrame(folds_checked)
display(fold_check_table)

# Confirm the final holdout remains a separate 42-calendar-day window.
holdout_start = pd.Timestamp("2015-06-20")
holdout_end = pd.Timestamp("2015-07-31")

assert (holdout_end - holdout_start).days == 41
assert features.loc[
    features["Date"].between(holdout_start, holdout_end)
].shape[0] > 0

print("Fold boundaries and training-label availability checks passed.")

In [ ]:
# Cell 30: Inspect the rolling feature alignment for the failing row

store_id = 4
target_date = pd.Timestamp("2014-02-06")
cutoff_date = target_date - pd.Timedelta(days=42)
window_start = cutoff_date - pd.Timedelta(days=56)

actual_row = features.loc[
    (features["Store"] == store_id)
    & (features["Date"] == target_date)
]

store_history = feature_data.loc[
    feature_data["Store"] == store_id
].sort_values("Date")

window_history = store_history.loc[
    (store_history["Date"] > window_start)
    & (store_history["Date"] <= cutoff_date),
    ["Date", "Sales"]
]

print("Target date:", target_date.date())
print("Required cutoff:", cutoff_date.date())
print("Rolling window:", window_start.date(), "to", cutoff_date.date())
print("Historical open-day rows in window:", len(window_history))
print("Expected rolling mean:", window_history["Sales"].mean())
print("Actual feature value:", actual_row["Sales_rolling_mean_56d"].iloc[0])

# Inspect the rolling feature's source date directly.
source_row = rolling_features.loc[
    (rolling_features["Store"] == store_id)
    & (rolling_features["Date"] == target_date)
]

print("\nRolling feature lookup row:")
display(source_row)

print("\nHistorical sales included in the expected window:")
display(window_history)

Target date: 2014-02-06
Required cutoff: 2013-12-26
Rolling window: 2013-10-31 to 2013-12-26
Historical open-day rows in window: 46
Expected rolling mean: 10311.41304347826
Actual feature value: nan

Rolling feature lookup row:


,Store,Date,Sales_rolling_mean_56d



Historical sales included in the expected window:


,Date,Sales
2598,2013-11-01,8792
2599,2013-11-02,10519
2600,2013-11-04,10203
2601,2013-11-05,9616
2602,2013-11-06,9236
2603,2013-11-07,8572
2604,2013-11-08,9997
2605,2013-11-09,10373
2606,2013-11-11,8241
2607,2013-11-12,7179


In [ ]:
# Cell 31: Rebuild rolling means on a daily calendar

rolling_parts = []

for store_id, group in feature_data.groupby("Store", sort=False):
    group = group.sort_values("Date").copy()

    # Create a continuous calendar, but keep unobserved/closed days as NaN.
    daily_sales = (
        group.set_index("Date")["Sales"]
        .reindex(
            pd.date_range(
                start=feature_data["Date"].min(),
                end=feature_data["Date"].max(),
                freq="D"
            )
        )
    )

    # Mean over the 56 calendar days ending on each source date.
    # NaNs are ignored; they are never interpreted as zero sales.
    rolling_mean = daily_sales.rolling(
        "56D",
        closed="right",
        min_periods=1
    ).mean()

    # A source-date value at s becomes the feature for target date s + 42.
    part = pd.DataFrame({
        "Store": store_id,
        "Date": rolling_mean.index + pd.Timedelta(days=42),
        "Sales_rolling_mean_56d": rolling_mean.to_numpy()
    })

    rolling_parts.append(part)

rolling_features = pd.concat(rolling_parts, ignore_index=True)

rolling_lookup = rolling_features.set_index(["Store", "Date"])[
    "Sales_rolling_mean_56d"
]

rolling_keys = pd.MultiIndex.from_arrays(
    [features["Store"], features["Date"]],
    names=["Store", "Date"]
)

features["Sales_rolling_mean_56d"] = rolling_keys.map(rolling_lookup)

print("Rolling feature rebuilt.")
print("Missing rolling means:", features["Sales_rolling_mean_56d"].isna().sum())

# Check the specific row that failed.
display(
    features.loc[
        (features["Store"] == 4)
        & (features["Date"] == pd.Timestamp("2014-02-06")),
        ["Store", "Date", "Sales_rolling_mean_56d"]
    ]
)

Rolling feature rebuilt.
Missing rolling means: 46428


,Store,Date,Sales_rolling_mean_56d
371127,4,2014-02-06,10311.413043


In [ ]:
# Cell 32: Verify rolling-mean values against the historical window

for _, row in sample.iterrows():
    store_id = row["Store"]
    target_date = pd.Timestamp(row["Date"])

    cutoff = target_date - pd.Timedelta(days=42)
    window_start = cutoff - pd.Timedelta(days=56)

    store_history = history_by_store[store_id]

    historical_window = store_history.loc[
        (store_history["Date"] > window_start)
        & (store_history["Date"] <= cutoff),
        "Sales"
    ]

    expected = (
        historical_window.mean()
        if len(historical_window) > 0
        else np.nan
    )

    actual = features.loc[
        (features["Store"] == store_id)
        & (features["Date"] == target_date),
        "Sales_rolling_mean_56d"
    ].iloc[0]

    assert (
        (pd.isna(expected) and pd.isna(actual))
        or (
            pd.notna(expected)
            and pd.notna(actual)
            and np.isclose(expected, actual)
        )
    ), (
        f"Rolling mean mismatch: Store {store_id}, "
        f"target {target_date.date()}, expected={expected}, actual={actual}"
    )

print(f"Rolling-mean provenance checks passed for {len(sample)} sampled rows.")

Rolling-mean provenance checks passed for 300 sampled rows.


In [ ]:
# Cell 33: Recheck feature provenance after repairing rolling alignment

rng = np.random.default_rng(42)
sample_indices = rng.choice(
    features.index.to_numpy(),
    size=min(300, len(features)),
    replace=False
)
sample = features.loc[sample_indices].copy()

history_by_store = {
    store_id: group.sort_values("Date")
    for store_id, group in feature_data.groupby("Store")
}

for _, row in sample.iterrows():
    store_id = row["Store"]
    target_date = pd.Timestamp(row["Date"])
    cutoff = target_date - pd.Timedelta(days=42)

    # Exact lags
    for lag in [42, 49, 364]:
        expected = expected_sales(
            store_id, target_date - pd.Timedelta(days=lag)
        )
        actual = row[f"Sales_lag_{lag}"]

        assert (
            (pd.isna(expected) and pd.isna(actual))
            or (
                pd.notna(expected)
                and pd.notna(actual)
                and np.isclose(expected, actual)
            )
        ), f"Lag {lag} mismatch: Store {store_id}, {target_date.date()}"

    # Mean of eight same-weekday observations
    values = [
        expected_sales(store_id, target_date - pd.Timedelta(days=lag))
        for lag in range(42, 92, 7)
    ]
    expected_mean = np.nanmean(values) if not np.isnan(values).all() else np.nan
    actual_mean = row["Sales_same_weekday_mean_8w"]

    assert (
        (pd.isna(expected_mean) and pd.isna(actual_mean))
        or (
            pd.notna(expected_mean)
            and pd.notna(actual_mean)
            and np.isclose(expected_mean, actual_mean)
        )
    ), f"Weekday mean mismatch: Store {store_id}, {target_date.date()}"

    # Rolling mean ending at the target's t-42 cutoff
    store_history = history_by_store[store_id]
    window_start = cutoff - pd.Timedelta(days=56)

    window_sales = store_history.loc[
        (store_history["Date"] > window_start)
        & (store_history["Date"] <= cutoff),
        "Sales"
    ]

    expected_rolling = window_sales.mean() if len(window_sales) else np.nan
    actual_rolling = row["Sales_rolling_mean_56d"]

    assert (
        (pd.isna(expected_rolling) and pd.isna(actual_rolling))
        or (
            pd.notna(expected_rolling)
            and pd.notna(actual_rolling)
            and np.isclose(expected_rolling, actual_rolling)
        )
    ), f"Rolling mean mismatch: Store {store_id}, {target_date.date()}"

print(f"All sales-feature provenance checks passed for {len(sample)} sampled rows.")

All sales-feature provenance checks passed for 300 sampled rows.


In [ ]:
# Cell 34: Define and audit allowed model features

excluded_columns = {
    "Sales",
    "Customers",
    "Date",
    "Open",
    "Store"  # Keep store identity out initially; use static store attributes instead.
}

feature_columns = [
    column for column in features.columns
    if column not in excluded_columns
]

forbidden_predictors = {"Sales", "Customers", "Open", "Date"}
assert not (set(feature_columns) & forbidden_predictors)

# All predictors must be numeric for this initial HistGradientBoosting model.
non_numeric = features[feature_columns].select_dtypes(
    exclude=[np.number, "bool"]
).columns.tolist()

print("Number of candidate predictors:", len(feature_columns))
print("Non-numeric predictors:", non_numeric)
print("Target column: Sales")
print("Customers included:", "Customers" in feature_columns)
print("Store ID included:", "Store" in feature_columns)

display(pd.DataFrame({"feature": feature_columns}))

Number of candidate predictors: 36
Non-numeric predictors: []
Target column: Sales
Customers included: False
Store ID included: False


,feature
0,Promo
1,SchoolHoliday
2,DayOfWeek
3,Month
4,DayOfMonth
5,WeekOfYear
6,IsWeekend
7,Sales_lag_42
8,Sales_lag_49
9,Sales_lag_364


Fold 1: training rows=649,493; validation rows=39,924; common comparison rows=30,832
Fold 2: training rows=689,417; validation rows=40,176; common comparison rows=39,703
Fold 3: training rows=729,593; validation rows=37,044; common comparison rows=33,600
Fold 4: training rows=766,637; validation rows=37,473; common comparison rows=31,279


,fold,method,eligible_targets,WAPE_pct,forecast_to_actual_ratio
0,1,HistGradientBoostingRegressor,30832,11.359712,1.071471
3,1,pred_weekday_mean,30832,17.031046,1.101614
2,1,pred_lag364,30832,20.378156,0.967363
1,1,pred_lag42,30832,28.590513,1.230181
4,2,HistGradientBoostingRegressor,39703,8.984187,0.987699
6,2,pred_lag364,39703,10.610229,0.968180
7,2,pred_weekday_mean,39703,18.615789,1.132313
5,2,pred_lag42,39703,19.822589,1.021971
8,3,HistGradientBoostingRegressor,33600,10.584053,0.990125
10,3,pred_lag364,33600,15.330880,0.975659


,method,mean_WAPE_pct,min_WAPE_pct,max_WAPE_pct,mean_forecast_to_actual_ratio
0,HistGradientBoostingRegressor,9.938539,8.826204,11.359712,1.007790
1,pred_lag364,14.654713,10.610229,20.378156,0.970752
3,pred_weekday_mean,17.264966,14.061549,19.351479,1.019771
2,pred_lag42,23.562249,18.216641,28.590513,1.038840


In [ ]:
# Cell 36: Analyze validation errors by business segment

validation_predictions = pd.concat(
    model_fold_predictions,
    ignore_index=True
)

# Attach known target-day attributes and store type for analysis.
analysis_columns = features[
    ["Store", "Date", "Promo", "SchoolHoliday"]
].drop_duplicates(["Store", "Date"])

validation_predictions = validation_predictions.merge(
    analysis_columns,
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

store_types = store_metadata[["Store", "StoreType"]].drop_duplicates("Store")

validation_predictions = validation_predictions.merge(
    store_types,
    on="Store",
    how="left",
    validate="many_to_one"
)

validation_predictions["abs_error"] = (
    validation_predictions["Sales"]
    - validation_predictions["model_prediction"]
).abs()

validation_predictions["signed_error"] = (
    validation_predictions["model_prediction"]
    - validation_predictions["Sales"]
)

def summarize_segment(data, segment_name):
    result = (
        data.groupby(segment_name, dropna=False)
        .agg(
            target_rows=("Sales", "size"),
            actual_sales=("Sales", "sum"),
            predicted_sales=("model_prediction", "sum"),
            absolute_error=("abs_error", "sum"),
            mean_signed_error=("signed_error", "mean")
        )
        .reset_index()
    )

    result["WAPE_pct"] = (
        100 * result["absolute_error"] / result["actual_sales"]
    )
    result["forecast_to_actual_ratio"] = (
        result["predicted_sales"] / result["actual_sales"]
    )

    return result.sort_values("WAPE_pct", ascending=False)

validation_predictions["Promo_status"] = (
    validation_predictions["Promo"]
    .map({0: "No promotion", 1: "Promotion"})
    .fillna("Unknown")
)

validation_predictions["SchoolHoliday_status"] = (
    validation_predictions["SchoolHoliday"]
    .map({0: "No school holiday", 1: "School holiday"})
    .fillna("Unknown")
)

validation_predictions["StateHoliday_status"] = (
    validation_predictions["Date"].map(
        feature_data.drop_duplicates(["Store", "Date"])
        .set_index(["Store", "Date"])["StateHoliday"]
        .to_dict()
    )
)

print("Error by promotion status")
display(summarize_segment(validation_predictions, "Promo_status"))

print("Error by store type")
display(summarize_segment(validation_predictions, "StoreType"))

print("Error by school-holiday status")
display(summarize_segment(validation_predictions, "SchoolHoliday_status"))

# Store-level error: require at least 100 evaluated target rows.
store_error = (
    validation_predictions.groupby("Store")
    .agg(
        target_rows=("Sales", "size"),
        actual_sales=("Sales", "sum"),
        predicted_sales=("model_prediction", "sum"),
        absolute_error=("abs_error", "sum"),
        mean_signed_error=("signed_error", "mean")
    )
    .reset_index()
)

store_error = store_error.loc[store_error["target_rows"] >= 100].copy()
store_error["WAPE_pct"] = (
    100 * store_error["absolute_error"] / store_error["actual_sales"]
)
store_error["forecast_to_actual_ratio"] = (
    store_error["predicted_sales"] / store_error["actual_sales"]
)

print("20 highest-WAPE stores with at least 100 evaluated rows")
display(store_error.sort_values("WAPE_pct", ascending=False).head(20))

Error by promotion status


,Promo_status,target_rows,actual_sales,predicted_sales,absolute_error,mean_signed_error,WAPE_pct,forecast_to_actual_ratio
0,No promotion,70493,426840027,4.184472e+08,4.428900e+07,-119.058875,10.376019,0.980337
1,Promotion,64921,537696804,5.513211e+08,5.115796e+07,209.859376,9.514277,1.025338


Error by store type


,StoreType,target_rows,actual_sales,predicted_sales,absolute_error,mean_signed_error,WAPE_pct,forecast_to_actual_ratio
2,c,18131,127718809,1.295734e+08,1.347132e+07,102.289327,10.547637,1.014521
0,a,73233,513426651,5.164831e+08,5.082821e+07,41.736503,9.899799,1.005953
1,b,2816,30417584,3.046417e+07,2.952728e+06,16.543428,9.707305,1.001532
3,d,41234,292973787,2.932476e+08,2.819472e+07,6.639664,9.623631,1.000934


Error by school-holiday status


,SchoolHoliday_status,target_rows,actual_sales,predicted_sales,absolute_error,mean_signed_error,WAPE_pct,forecast_to_actual_ratio
1,School holiday,15273,120722417,1.179569e+08,1.496778e+07,-181.073327,12.398511,0.977092
0,No school holiday,120141,843814414,8.518114e+08,8.047918e+07,66.563423,9.537546,1.009477


20 highest-WAPE stores with at least 100 evaluated rows


,Store,target_rows,actual_sales,predicted_sales,absolute_error,mean_signed_error,WAPE_pct,forecast_to_actual_ratio
1080,1081,168,897354,9.870906e+05,243287.848629,534.146299,27.111692,1.100001
529,530,147,607899,5.464288e+05,162127.801418,-418.164342,26.670187,0.898881
575,576,128,997285,9.819198e+05,261904.719656,-120.040391,26.261773,0.984593
467,468,128,744500,8.451240e+05,161432.545896,786.124815,21.683351,1.135156
838,839,128,534519,6.268249e+05,105667.877716,721.140178,19.768779,1.172690
731,732,143,755444,7.111759e+05,143005.408446,-309.567073,18.929981,0.941401
298,299,144,864423,8.202066e+05,159950.128281,-307.058395,18.503687,0.948849
769,770,126,785167,8.669427e+05,142235.556624,649.013364,18.115325,1.104151
38,39,126,685318,6.401966e+05,123844.313391,-358.106731,18.071073,0.934160
197,198,128,360871,3.931732e+05,62709.426017,252.361022,17.377242,1.089512


In [ ]:
# Cell 37: Investigate worst-store errors and state-holiday performance

# Attach StateHoliday from the original open-day records.
holiday_lookup = feature_data[
    ["Store", "Date", "StateHoliday"]
].drop_duplicates(["Store", "Date"])

diagnostics = validation_predictions.drop(
    columns=["StateHoliday_status"], errors="ignore"
).merge(
    holiday_lookup,
    on=["Store", "Date"],
    how="left",
    validate="many_to_one"
)

diagnostics["StateHoliday_status"] = diagnostics["StateHoliday"].replace({
    "0": "No state holiday",
    "a": "Public holiday",
    "b": "Easter holiday",
    "c": "Christmas"
}).fillna("Unknown")

print("Error by state-holiday status")
display(summarize_segment(diagnostics, "StateHoliday_status"))

# Examine the 10 worst stores, including fold-level performance.
worst_store_ids = (
    store_error.sort_values("WAPE_pct", ascending=False)
    .head(10)["Store"]
    .tolist()
)

worst_store_fold = (
    diagnostics[diagnostics["Store"].isin(worst_store_ids)]
    .groupby(["Store", "fold"])
    .agg(
        target_rows=("Sales", "size"),
        actual_sales=("Sales", "sum"),
        predicted_sales=("model_prediction", "sum"),
        absolute_error=("abs_error", "sum")
    )
    .reset_index()
)

worst_store_fold["WAPE_pct"] = (
    100 * worst_store_fold["absolute_error"]
    / worst_store_fold["actual_sales"]
)

print("Fold-level performance for the 10 worst stores")
display(worst_store_fold.sort_values(["Store", "fold"]))

# Check the original record coverage and open-day zero sales for those stores.
store_history_check = (
    feature_data[feature_data["Store"].isin(worst_store_ids)]
    .groupby("Store")
    .agg(
        observed_open_days=("Sales", "size"),
        first_observed_date=("Date", "min"),
        last_observed_date=("Date", "max"),
        zero_sales_open_days=("Sales", lambda s: int((s == 0).sum()))
    )
    .reset_index()
)

print("Historical coverage for the 10 worst stores")
display(store_history_check.sort_values("Store"))

Error by state-holiday status


,StateHoliday_status,target_rows,actual_sales,predicted_sales,absolute_error,mean_signed_error,WAPE_pct,forecast_to_actual_ratio
0,Easter holiday,49,515394,4.664383e+05,1.535012e+05,-999.096164,29.783275,0.905013
2,Public holiday,174,1616054,1.460702e+06,3.509145e+05,-892.825433,21.714279,0.903870
1,No state holiday,135191,962405383,9.678412e+08,9.494255e+07,40.208081,9.865131,1.005648


Fold-level performance for the 10 worst stores


,Store,fold,target_rows,actual_sales,predicted_sales,absolute_error,WAPE_pct
0,39,1,33,157084,170514.785319,19147.648557,12.189433
1,39,2,36,164708,164150.940991,15080.600282,9.155961
2,39,3,30,211393,160547.168419,51656.637877,24.436305
3,39,4,27,152133,144983.657151,37959.426675,24.951474
4,198,1,33,92686,108256.425443,20159.327467,21.750132
5,198,2,36,91015,104384.740270,18622.692664,20.461125
6,198,3,30,94722,95537.921433,14201.926978,14.993272
7,198,4,29,82448,84994.123669,9725.478908,11.795894
8,299,1,34,164903,200077.030556,40695.519856,24.678459
9,299,2,36,191365,188730.436611,22722.240777,11.873770


Historical coverage for the 10 worst stores


,Store,observed_open_days,first_observed_date,last_observed_date,zero_sales_open_days
0,39,779,2013-01-02,2015-07-31,0
1,198,782,2013-01-02,2015-07-31,0
2,299,882,2013-01-02,2015-07-31,0
3,468,784,2013-01-02,2015-07-31,0
4,530,906,2013-01-01,2015-07-31,0
5,576,784,2013-01-02,2015-07-31,0
6,732,883,2013-01-02,2015-07-31,0
7,770,779,2013-01-02,2015-07-31,0
8,839,784,2013-01-02,2015-07-31,0
9,1081,757,2013-07-05,2015-07-31,0


In [ ]:
# Cell 38: Calculate out-of-fold multiplicative residual ratios

validation_predictions = pd.concat(
    model_fold_predictions,
    ignore_index=True
).copy()

# Exclude zero actuals from ratio calculation because actual/predicted
# ratios are not informative in the same way when actual sales are zero.
ratio_data = validation_predictions.loc[
    (validation_predictions["Sales"] > 0)
    & (validation_predictions["model_prediction"] > 0)
].copy()

ratio_data["actual_to_predicted_ratio"] = (
    ratio_data["Sales"] / ratio_data["model_prediction"]
)

print("Validation prediction rows:", len(validation_predictions))
print("Rows used for ratio-based bands:", len(ratio_data))
print("Excluded rows:", len(validation_predictions) - len(ratio_data))

display(
    ratio_data["actual_to_predicted_ratio"]
    .describe(percentiles=[0.025, 0.05, 0.95, 0.975])
    .to_frame("actual_to_predicted_ratio")
)

Validation prediction rows: 135414
Rows used for ratio-based bands: 135413
Excluded rows: 1


,actual_to_predicted_ratio
count,135413.000000
mean,0.998817
std,0.140320
min,0.047103
2.5%,0.757581
5%,0.799203
95%,1.231859
97.5%,1.308157
max,4.484454


In [ ]:
# Cell 39: Estimate multiplicative bands and measure empirical coverage

lower_90, upper_90 = ratio_data[
    "actual_to_predicted_ratio"
].quantile([0.05, 0.95])

lower_95, upper_95 = ratio_data[
    "actual_to_predicted_ratio"
].quantile([0.025, 0.975])

coverage_data = validation_predictions.copy()

coverage_data["lower_90"] = (
    coverage_data["model_prediction"] * lower_90
).clip(lower=0)
coverage_data["upper_90"] = (
    coverage_data["model_prediction"] * upper_90
).clip(lower=0)

coverage_data["lower_95"] = (
    coverage_data["model_prediction"] * lower_95
).clip(lower=0)
coverage_data["upper_95"] = (
    coverage_data["model_prediction"] * upper_95
).clip(lower=0)

coverage_data["covered_90"] = (
    (coverage_data["Sales"] >= coverage_data["lower_90"])
    & (coverage_data["Sales"] <= coverage_data["upper_90"])
)

coverage_data["covered_95"] = (
    (coverage_data["Sales"] >= coverage_data["lower_95"])
    & (coverage_data["Sales"] <= coverage_data["upper_95"])
)

coverage_summary = pd.DataFrame([
    {
        "band": "Nominal 90%",
        "lower_ratio": lower_90,
        "upper_ratio": upper_90,
        "observed_coverage_pct": 100 * coverage_data["covered_90"].mean()
    },
    {
        "band": "Nominal 95%",
        "lower_ratio": lower_95,
        "upper_ratio": upper_95,
        "observed_coverage_pct": 100 * coverage_data["covered_95"].mean()
    }
])

display(coverage_summary)

,band,lower_ratio,upper_ratio,observed_coverage_pct
0,Nominal 90%,0.799203,1.231859,89.998818
1,Nominal 95%,0.757581,1.308157,94.998302


In [ ]:
# Cell 40: Verify the final holdout is untouched before evaluation

holdout_start = pd.Timestamp("2015-06-20")
holdout_end = pd.Timestamp("2015-07-31")

holdout_rows = features.loc[
    features["Date"].between(holdout_start, holdout_end)
].copy()

holdout_origin = holdout_start - pd.Timedelta(days=1)
final_train_rows = features.loc[
    features["Date"] <= holdout_origin
].copy()

assert len(holdout_rows) > 0
assert final_train_rows["Date"].max() == holdout_origin
assert holdout_rows["Date"].min() == holdout_start
assert holdout_rows["Date"].max() == holdout_end
assert (holdout_end - holdout_start).days == 41

print("Holdout dates:", holdout_start.date(), "to", holdout_end.date())
print("Holdout open store-days:", f"{len(holdout_rows):,}")
print("Final training-label cutoff:", final_train_rows["Date"].max().date())
print("Holdout remains separate from training labels.")

Holdout dates: 2015-06-20 to 2015-07-31
Holdout open store-days: 40,282
Final training-label cutoff: 2015-06-19
Holdout remains separate from training labels.


In [ ]:
# Cell 41: One-time final holdout evaluation

from sklearn.ensemble import HistGradientBoostingRegressor

# Fit only on labels available before the holdout starts.
final_train_rows = features.loc[
    features["Date"] < holdout_start
].copy()

final_holdout_rows = features.loc[
    features["Date"].between(holdout_start, holdout_end)
].copy()

final_model = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=200,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

final_model.fit(
    final_train_rows[feature_columns],
    final_train_rows["Sales"]
)

final_holdout_rows["model_prediction"] = final_model.predict(
    final_holdout_rows[feature_columns]
).clip(min=0)

# Build baseline predictions using the same historical sales lookup.
holdout_baselines = predict_baselines(
    history=sales_data.loc[sales_data["Date"] < holdout_start],
    targets=final_holdout_rows[["Store", "Date", "Sales"]]
)

holdout_comparison = holdout_baselines.merge(
    final_holdout_rows[["Store", "Date", "model_prediction"]],
    on=["Store", "Date"],
    how="inner",
    validate="one_to_one"
)

# Fair comparison: same holdout rows must have all baseline predictions.
holdout_comparison = holdout_comparison.dropna(
    subset=[
        "model_prediction",
        "pred_lag42",
        "pred_lag364",
        "pred_weekday_mean"
    ]
).copy()

actual = holdout_comparison["Sales"].to_numpy(dtype=float)
total_actual = actual.sum()

holdout_methods = {
    "HistGradientBoostingRegressor":
        holdout_comparison["model_prediction"].to_numpy(dtype=float),
    "pred_lag42":
        holdout_comparison["pred_lag42"].to_numpy(dtype=float),
    "pred_lag364":
        holdout_comparison["pred_lag364"].to_numpy(dtype=float),
    "pred_weekday_mean":
        holdout_comparison["pred_weekday_mean"].to_numpy(dtype=float)
}

holdout_results = []

for method_name, predictions in holdout_methods.items():
    holdout_results.append({
        "method": method_name,
        "eligible_targets": len(holdout_comparison),
        "WAPE_pct": wape(actual, predictions),
        "forecast_to_actual_ratio": predictions.sum() / total_actual,
        "bias_pct": 100 * (predictions.sum() - total_actual) / total_actual
    })

holdout_results_df = pd.DataFrame(holdout_results).sort_values("WAPE_pct")

# Evaluate frozen multiplicative bands on the model's holdout predictions.
model_predictions = holdout_comparison["model_prediction"].to_numpy(dtype=float)

holdout_comparison["lower_90"] = model_predictions * lower_90
holdout_comparison["upper_90"] = model_predictions * upper_90
holdout_comparison["lower_95"] = model_predictions * lower_95
holdout_comparison["upper_95"] = model_predictions * upper_95

holdout_comparison["covered_90"] = (
    (actual >= holdout_comparison["lower_90"].to_numpy())
    & (actual <= holdout_comparison["upper_90"].to_numpy())
)

holdout_comparison["covered_95"] = (
    (actual >= holdout_comparison["lower_95"].to_numpy())
    & (actual <= holdout_comparison["upper_95"].to_numpy())
)

holdout_coverage = pd.DataFrame([
    {
        "band": "Nominal 90%",
        "lower_ratio": lower_90,
        "upper_ratio": upper_90,
        "observed_holdout_coverage_pct":
            100 * holdout_comparison["covered_90"].mean()
    },
    {
        "band": "Nominal 95%",
        "lower_ratio": lower_95,
        "upper_ratio": upper_95,
        "observed_holdout_coverage_pct":
            100 * holdout_comparison["covered_95"].mean()
    }
])

print("Training rows:", f"{len(final_train_rows):,}")
print("Holdout open store-days:", f"{len(final_holdout_rows):,}")
print("Common comparison rows:", f"{len(holdout_comparison):,}")

print("\nFinal holdout point-forecast comparison")
display(holdout_results_df)

print("\nFrozen prediction-band coverage on holdout")
display(holdout_coverage)

KeyError: "['Sales_rolling_mean_56d', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'StateHoliday_0', 'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c', 'StateHoliday_nan', 'StoreType_a', 'StoreType_b', 'StoreType_c', 'StoreType_d', 'StoreType_nan', 'Assortment_a', 'Assortment_b', 'Assortment_c', 'Assortment_nan', 'PromoInterval_Feb,May,Aug,Nov', 'PromoInterval_Jan,Apr,Jul,Oct', 'PromoInterval_Mar,Jun,Sept,Dec', 'PromoInterval_nan'] not in index"

In [ ]:
# Cell 41A: Diagnose missing model features before holdout evaluation

print("features shape:", features.shape)
print("features columns:", len(features.columns))
print("feature_columns count:", len(feature_columns))

missing_columns = [
    column for column in feature_columns
    if column not in features.columns
]

print("\nMissing model columns:")
print(missing_columns)

print("\nDoes features contain Sales?", "Sales" in features.columns)
print("\nFirst 15 feature table columns:")
print(features.columns[:15].tolist())

features shape: (844392, 15)
features columns: 15
feature_columns count: 36

Missing model columns:
['Sales_rolling_mean_56d', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'StateHoliday_0', 'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c', 'StateHoliday_nan', 'StoreType_a', 'StoreType_b', 'StoreType_c', 'StoreType_d', 'StoreType_nan', 'Assortment_a', 'Assortment_b', 'Assortment_c', 'Assortment_nan', 'PromoInterval_Feb,May,Aug,Nov', 'PromoInterval_Jan,Apr,Jul,Oct', 'PromoInterval_Mar,Jun,Sept,Dec', 'PromoInterval_nan']

Does features contain Sales? True

First 15 feature table columns:
['Store', 'Date', 'Sales', 'Promo', 'StateHoliday', 'SchoolHoliday', 'DayOfWeek', 'Month', 'DayOfMonth', 'WeekOfYear', 'IsWeekend', 'Sales_lag_42', 'Sales_lag_49', 'Sales_lag_364', 'Sales_same_weekday_mean_8w']


In [ ]:
# Cell 41B: Rebuild the complete feature table safely

# 1. Start from the original open-store-day data
feature_data = feature_data.copy()
feature_data["Date"] = pd.to_datetime(feature_data["Date"])
feature_data["StateHoliday"] = feature_data["StateHoliday"].astype(str)
feature_data = feature_data.sort_values(["Store", "Date"]).reset_index(drop=True)

# 2. Rebuild target and calendar features
features = feature_data[
    ["Store", "Date", "Sales", "Promo", "StateHoliday", "SchoolHoliday"]
].copy()

features["DayOfWeek"] = features["Date"].dt.dayofweek + 1
features["Month"] = features["Date"].dt.month
features["DayOfMonth"] = features["Date"].dt.day
features["WeekOfYear"] = features["Date"].dt.isocalendar().week.astype(int)
features["IsWeekend"] = (features["Date"].dt.dayofweek >= 5).astype(int)

# 3. Create exact-date sales lookup for each store
sales_lookup = feature_data.set_index(["Store", "Date"])["Sales"].sort_index()

def lookup_sales(store, date):
    try:
        return sales_lookup.loc[(store, date)]
    except KeyError:
        return np.nan

for lag in [42, 49, 364]:
    features[f"Sales_lag_{lag}"] = [
        lookup_sales(store, date - pd.Timedelta(days=lag))
        for store, date in zip(features["Store"], features["Date"])
    ]

# 4. Same-weekday mean over the 8 dates ending 42 days before target
weekday_means = []

for store, date in zip(features["Store"], features["Date"]):
    prior_dates = [
        date - pd.Timedelta(days=lag)
        for lag in range(42, 92, 7)
    ]
    values = [lookup_sales(store, prior_date) for prior_date in prior_dates]
    weekday_means.append(np.nanmean(values) if not np.isnan(values).all() else np.nan)

features["Sales_same_weekday_mean_8w"] = weekday_means

# 5. Rolling mean over (target date - 98 days, target date - 42 days]
rolling_parts = []

for store, group in feature_data.groupby("Store", sort=False):
    daily_sales = group.set_index("Date")["Sales"].sort_index()

    # Missing dates remain NaN; they are NOT treated as zero sales.
    daily_calendar = daily_sales.reindex(
        pd.date_range(daily_sales.index.min(), daily_sales.index.max(), freq="D")
    )

    rolling_mean = daily_calendar.rolling(
        "56D", closed="right", min_periods=1
    ).mean()

    # A value calculated at cutoff date is available to target date cutoff + 42 days.
    rolling_mean.index = rolling_mean.index + pd.Timedelta(days=42)

    rolling_parts.append(
        pd.DataFrame({
            "Store": store,
            "Date": rolling_mean.index,
            "Sales_rolling_mean_56d": rolling_mean.values
        })
    )

rolling_features = pd.concat(rolling_parts, ignore_index=True)

features = features.merge(
    rolling_features,
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

# 6. Merge static store metadata
store_metadata = store.drop_duplicates("Store").copy()

features = features.merge(
    store_metadata,
    on="Store",
    how="left",
    validate="many_to_one",
    suffixes=("", "_store")
)

# 7. One-hot encode categorical columns
categorical_columns = [
    column for column in ["StateHoliday", "StoreType", "Assortment", "PromoInterval"]
    if column in features.columns
]

features[categorical_columns] = features[categorical_columns].astype("string")

features = pd.get_dummies(
    features,
    columns=categorical_columns,
    dummy_na=True,
    dtype=int
)

# 8. Recreate the predictor list; exclude target and identifiers
excluded_columns = {"Sales", "Customers", "Date", "Open", "Store"}
feature_columns = [
    column for column in features.columns
    if column not in excluded_columns
]

print("Rebuilt features shape:", features.shape)
print("Predictor count:", len(feature_columns))
print("Missing predictors:", [
    column for column in feature_columns if column not in features.columns
])
print("Duplicate Store-Date rows:", features.duplicated(["Store", "Date"]).sum())
print("Non-numeric predictors:", features[feature_columns].select_dtypes(exclude="number").columns.tolist())

assert features.duplicated(["Store", "Date"]).sum() == 0
assert not [column for column in feature_columns if column not in features.columns]
assert features[feature_columns].select_dtypes(exclude="number").shape[1] == 0
assert "Customers" not in feature_columns
assert features.shape[0] == feature_data.shape[0]

print("Feature table rebuilt and basic checks passed.")

KeyboardInterrupt: 

In [ ]:
# Cell 41C: Check notebook state after interrupting the slow rebuild

print("features shape:", features.shape)
print("feature_data shape:", feature_data.shape)
print("Store metadata shape:", store.shape)
print("Kernel variables still available:", all(
    name in globals()
    for name in ["feature_data", "store", "feature_columns", "validation_predictions"]
))

features shape: (844392, 13)
feature_data shape: (844392, 9)
Store metadata shape: (1115, 10)
Kernel variables still available: True


In [ ]:
# Cell 41D: Restore base features using vectorized operations

feature_data = feature_data.copy()
feature_data["Date"] = pd.to_datetime(feature_data["Date"])
feature_data["StateHoliday"] = feature_data["StateHoliday"].astype(str)
feature_data = feature_data.sort_values(["Store", "Date"]).reset_index(drop=True)

features = feature_data[
    ["Store", "Date", "Sales", "Promo", "StateHoliday", "SchoolHoliday"]
].copy()

features["DayOfWeek"] = features["Date"].dt.dayofweek + 1
features["Month"] = features["Date"].dt.month
features["DayOfMonth"] = features["Date"].dt.day
features["WeekOfYear"] = features["Date"].dt.isocalendar().week.astype("int16")
features["IsWeekend"] = (features["Date"].dt.dayofweek >= 5).astype("int8")

# Vectorized exact-date lags; no Python loop over 844,392 rows
sales_lookup = feature_data.set_index(["Store", "Date"])["Sales"]

for lag in [42, 49, 364]:
    lookup_index = pd.MultiIndex.from_arrays([
        features["Store"].to_numpy(),
        (features["Date"] - pd.Timedelta(days=lag)).to_numpy()
    ], names=["Store", "Date"])

    features[f"Sales_lag_{lag}"] = sales_lookup.reindex(lookup_index).to_numpy()

print("Base features restored:", features.shape)
print(features.columns.tolist())

Base features restored: (844392, 14)
['Store', 'Date', 'Sales', 'Promo', 'StateHoliday', 'SchoolHoliday', 'DayOfWeek', 'Month', 'DayOfMonth', 'WeekOfYear', 'IsWeekend', 'Sales_lag_42', 'Sales_lag_49', 'Sales_lag_364']


In [ ]:
# Cell 41E: Restore same-weekday mean and rolling mean efficiently

# 1. Same-weekday mean: sales at t-42, t-49, ..., t-91
weekday_lags = [42, 49, 56, 63, 70, 77, 84, 91]
weekday_columns = []

for lag in weekday_lags:
    column = f"_weekday_lag_{lag}"
    lookup_index = pd.MultiIndex.from_arrays([
        features["Store"].to_numpy(),
        (features["Date"] - pd.Timedelta(days=lag)).to_numpy()
    ], names=["Store", "Date"])

    features[column] = sales_lookup.reindex(lookup_index).to_numpy()
    weekday_columns.append(column)

features["Sales_same_weekday_mean_8w"] = features[weekday_columns].mean(axis=1)
features.drop(columns=weekday_columns, inplace=True)

# 2. Rolling mean over the 56 calendar days ending at t-42.
# Reindex to daily calendars so missing/closed dates are NaN, not zero.
rolling_parts = []

for store_id, group in feature_data.groupby("Store", sort=False):
    daily = group.set_index("Date")["Sales"].sort_index()
    calendar = daily.reindex(
        pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    )
    rolling = calendar.rolling("56D", closed="right", min_periods=1).mean()
    rolling.index = rolling.index + pd.Timedelta(days=42)

    rolling_parts.append(pd.DataFrame({
        "Store": store_id,
        "Date": rolling.index,
        "Sales_rolling_mean_56d": rolling.to_numpy()
    }))

rolling_features = pd.concat(rolling_parts, ignore_index=True)

features = features.merge(
    rolling_features,
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

print("Feature table shape:", features.shape)
print("Same-weekday mean missing:", features["Sales_same_weekday_mean_8w"].isna().sum())
print("Rolling mean missing:", features["Sales_rolling_mean_56d"].isna().sum())

Feature table shape: (844392, 16)
Same-weekday mean missing: 46866
Rolling mean missing: 46428


In [ ]:
# Cell 41F: Restore store metadata and categorical predictors

store_metadata = store.drop_duplicates("Store").copy()

features = features.merge(
    store_metadata,
    on="Store",
    how="left",
    validate="many_to_one",
    suffixes=("", "_store")
)

categorical_columns = [
    col for col in ["StateHoliday", "StoreType", "Assortment", "PromoInterval"]
    if col in features.columns
]

for col in categorical_columns:
    features[col] = features[col].astype("string")

features = pd.get_dummies(
    features,
    columns=categorical_columns,
    dummy_na=True,
    dtype=int
)

excluded_columns = {"Sales", "Customers", "Date", "Open", "Store"}
feature_columns = [
    col for col in features.columns
    if col not in excluded_columns
]

print("Final feature table shape:", features.shape)
print("Predictor count:", len(feature_columns))
print("Missing predictors:", [
    col for col in feature_columns if col not in features.columns
])
print("Non-numeric predictors:",
      features[feature_columns].select_dtypes(exclude="number").columns.tolist())
print("Duplicate Store-Date rows:",
      features.duplicated(["Store", "Date"]).sum())
print("Customers included:", "Customers" in feature_columns)

assert features.duplicated(["Store", "Date"]).sum() == 0
assert not features[feature_columns].select_dtypes(exclude="number").columns.tolist()
assert "Customers" not in feature_columns

print("Basic feature-table checks passed.")

Final feature table shape: (844392, 39)
Predictor count: 36
Missing predictors: []
Non-numeric predictors: []
Duplicate Store-Date rows: 0
Customers included: False
Basic feature-table checks passed.


In [ ]:
# Cell 41G: Verify rebuilt feature provenance and holdout separation

rng = np.random.default_rng(42)
sample = features.sample(n=300, random_state=42).copy()

# Lookup containing only observed open-day sales
history_by_store = {
    store_id: group.set_index("Date")["Sales"].sort_index()
    for store_id, group in feature_data.groupby("Store", sort=False)
}

def expected_sales(store_id, date):
    series = history_by_store.get(store_id)
    if series is None:
        return np.nan
    return series.get(date, np.nan)

# Check exact-date lags
for lag in [42, 49, 364]:
    expected = np.array([
        expected_sales(store_id, date - pd.Timedelta(days=lag))
        for store_id, date in zip(sample["Store"], sample["Date"])
    ], dtype=float)

    actual = sample[f"Sales_lag_{lag}"].to_numpy(dtype=float)
    np.testing.assert_allclose(actual, expected, rtol=1e-10, atol=1e-8,
                               equal_nan=True)

# Check the same-weekday mean
expected_means = []

for store_id, date in zip(sample["Store"], sample["Date"]):
    values = [
        expected_sales(store_id, date - pd.Timedelta(days=lag))
        for lag in [42, 49, 56, 63, 70, 77, 84, 91]
    ]
    expected_means.append(np.nanmean(values) if not np.isnan(values).all() else np.nan)

np.testing.assert_allclose(
    sample["Sales_same_weekday_mean_8w"].to_numpy(dtype=float),
    np.array(expected_means, dtype=float),
    rtol=1e-10, atol=1e-8, equal_nan=True
)

# Verify rolling feature using only dates strictly before target_date - 42 days,
# with the same 56-calendar-day window convention.
rolling_expected = []

for store_id, target_date in zip(sample["Store"], sample["Date"]):
    cutoff = target_date - pd.Timedelta(days=42)
    window_start = cutoff - pd.Timedelta(days=56)
    series = history_by_store[store_id]

    values = series.loc[(series.index > window_start) & (series.index <= cutoff)]
    rolling_expected.append(values.mean() if len(values) else np.nan)

np.testing.assert_allclose(
    sample["Sales_rolling_mean_56d"].to_numpy(dtype=float),
    np.array(rolling_expected, dtype=float),
    rtol=1e-10, atol=1e-8, equal_nan=True
)

# Confirm holdout rows are still separate from training labels
holdout_start = pd.Timestamp("2015-06-20")
holdout_end = pd.Timestamp("2015-07-31")
holdout_rows = features.loc[features["Date"].between(holdout_start, holdout_end)]
final_train_rows = features.loc[features["Date"] < holdout_start]

assert final_train_rows["Date"].max() == pd.Timestamp("2015-06-19")
assert holdout_rows["Date"].min() == holdout_start
assert holdout_rows["Date"].max() == holdout_end

print("PASS: exact-date lags match source sales.")
print("PASS: same-weekday means match source sales.")
print("PASS: rolling means respect the 42-day embargo.")
print("PASS: training labels stop on 2015-06-19.")
print("Holdout rows available, not scored:", len(holdout_rows))

PASS: exact-date lags match source sales.
PASS: same-weekday means match source sales.
PASS: rolling means respect the 42-day embargo.
PASS: training labels stop on 2015-06-19.
Holdout rows available, not scored: 40282


In [ ]:
# Cell 42: One-time final holdout evaluation

import json
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor

# Freeze the evaluation window
holdout_start = pd.Timestamp("2015-06-20")
holdout_end = pd.Timestamp("2015-07-31")
holdout_origin = holdout_start - pd.Timedelta(days=1)

final_train_rows = features.loc[features["Date"] <= holdout_origin].copy()
final_holdout_rows = features.loc[
    features["Date"].between(holdout_start, holdout_end)
].copy()

assert final_train_rows["Date"].max() == holdout_origin
assert final_holdout_rows["Date"].min() == holdout_start
assert final_holdout_rows["Date"].max() == holdout_end
assert len(final_holdout_rows) == 40282

# Fit the already-selected model; do not tune on holdout
final_model = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=200,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

final_model.fit(
    final_train_rows[feature_columns],
    final_train_rows["Sales"]
)

# Predict the holdout once
holdout_predictions = final_holdout_rows[
    ["Store", "Date", "Sales", "Promo", "SchoolHoliday"]
].copy()

holdout_predictions["model_prediction"] = final_model.predict(
    final_holdout_rows[feature_columns]
)

# Baselines use historical sales; target-date sales are not used as predictors
holdout_baselines = predict_baselines(
    sales_data.loc[sales_data["Date"] < holdout_start].copy(),
    holdout_predictions[["Store", "Date"]].copy()
)

holdout_predictions = holdout_predictions.merge(
    holdout_baselines,
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

# Model performance on every holdout target
model_wape_all = wape(
    holdout_predictions["Sales"],
    holdout_predictions["model_prediction"]
)

model_bias_all = (
    holdout_predictions["model_prediction"].sum()
    / holdout_predictions["Sales"].sum()
)

# Fair baseline comparison: identical store-days for every method
baseline_columns = [
    "pred_lag42",
    "pred_lag364",
    "pred_weekday_mean"
]

common_mask = holdout_predictions[
    ["Sales", "model_prediction"] + baseline_columns
].notna().all(axis=1)

comparison = holdout_predictions.loc[common_mask].copy()

comparison_metrics = {}
for column in ["model_prediction"] + baseline_columns:
    comparison_metrics[column] = {
        "wape_pct": float(wape(comparison["Sales"], comparison[column])),
        "forecast_actual_ratio": float(
            comparison[column].sum() / comparison["Sales"].sum()
        )
    }

# Compare the model with the previously selected best baseline
model_common_wape = comparison_metrics["model_prediction"]["wape_pct"]
best_baseline_wape = comparison_metrics["pred_lag364"]["wape_pct"]

improvement_pp = best_baseline_wape - model_common_wape

# Apply the previously frozen validation-derived multiplicative bands
holdout_predictions["lower_90"] = (
    holdout_predictions["model_prediction"] * lower_90
)
holdout_predictions["upper_90"] = (
    holdout_predictions["model_prediction"] * upper_90
)
holdout_predictions["lower_95"] = (
    holdout_predictions["model_prediction"] * lower_95
)
holdout_predictions["upper_95"] = (
    holdout_predictions["model_prediction"] * upper_95
)

actual = holdout_predictions["Sales"]
coverage_90 = (
    (actual >= holdout_predictions["lower_90"])
    & (actual <= holdout_predictions["upper_90"])
).mean() * 100

coverage_95 = (
    (actual >= holdout_predictions["lower_95"])
    & (actual <= holdout_predictions["upper_95"])
).mean() * 100

print("HOLDOUT RESULTS — FINAL EVALUATION")
print(f"Holdout rows: {len(holdout_predictions):,}")
print(f"Common comparison rows: {len(comparison):,}")
print(f"Model WAPE, all holdout rows: {model_wape_all:.3f}%")
print(f"Model WAPE, common rows: {model_common_wape:.3f}%")
print(f"Lag-364 baseline WAPE, common rows: {best_baseline_wape:.3f}%")
print(f"Improvement over lag-364 baseline: {improvement_pp:.3f} percentage points")
print(f"Model forecast/actual ratio, all rows: {model_bias_all:.4f}")
print("\nAll methods on the same eligible rows:")
for name, metrics in comparison_metrics.items():
    print(
        f"{name}: WAPE={metrics['wape_pct']:.3f}%, "
        f"forecast/actual={metrics['forecast_actual_ratio']:.4f}"
    )

print(f"\nEmpirical 90% band holdout coverage: {coverage_90:.2f}%")
print(f"Empirical 95% band holdout coverage: {coverage_95:.2f}%")

# Save the exact scored predictions so the evaluation is auditable
output_dir = Path("../reports")
output_dir.mkdir(parents=True, exist_ok=True)

holdout_predictions.to_csv(
    output_dir / "holdout_predictions.csv",
    index=False
)

print("\nSaved holdout predictions to reports/holdout_predictions.csv")

KeyError: "['Sales'] not in index"

In [ ]:
# Cell 42A: Fix baseline inputs for final holdout evaluation

# The baseline function expects Sales in targets, even though it uses
# historical Sales to generate predictions. Supply the actual column.
baseline_targets = final_holdout_rows[["Store", "Date", "Sales"]].copy()

holdout_baselines = predict_baselines(
    sales_data.loc[sales_data["Date"] < holdout_start].copy(),
    baseline_targets
)

# Keep the existing holdout_predictions table and merge baseline predictions
baseline_prediction_columns = [
    col for col in holdout_baselines.columns
    if col not in ["Store", "Date", "Sales"]
]

holdout_predictions = holdout_predictions.drop(
    columns=[col for col in baseline_prediction_columns
             if col in holdout_predictions.columns],
    errors="ignore"
)

holdout_predictions = holdout_predictions.merge(
    holdout_baselines[["Store", "Date"] + baseline_prediction_columns],
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

print("Holdout baseline predictions generated.")
print("Holdout rows:", len(holdout_predictions))
print("Baseline columns:", baseline_prediction_columns)

Holdout baseline predictions generated.
Holdout rows: 40282
Baseline columns: ['pred_lag42', 'pred_lag364', 'pred_weekday_mean']


In [ ]:
# Cell 42B: Calculate final holdout metrics

# Overall model performance
model_wape_all = wape(
    holdout_predictions["Sales"],
    holdout_predictions["model_prediction"]
)

model_bias_all = (
    holdout_predictions["model_prediction"].sum()
    / holdout_predictions["Sales"].sum()
)

# Compare all methods on exactly the same eligible rows
baseline_columns = ["pred_lag42", "pred_lag364", "pred_weekday_mean"]

common_mask = holdout_predictions[
    ["Sales", "model_prediction"] + baseline_columns
].notna().all(axis=1)

comparison = holdout_predictions.loc[common_mask].copy()

comparison_metrics = {}
for column in ["model_prediction"] + baseline_columns:
    comparison_metrics[column] = {
        "wape_pct": float(wape(comparison["Sales"], comparison[column])),
        "forecast_actual_ratio": float(
            comparison[column].sum() / comparison["Sales"].sum()
        )
    }

model_common_wape = comparison_metrics["model_prediction"]["wape_pct"]
best_baseline_wape = comparison_metrics["pred_lag364"]["wape_pct"]
improvement_pp = best_baseline_wape - model_common_wape

# Evaluate previously calibrated prediction bands
actual = holdout_predictions["Sales"]

coverage_90 = (
    (actual >= holdout_predictions["model_prediction"] * lower_90)
    & (actual <= holdout_predictions["model_prediction"] * upper_90)
).mean() * 100

coverage_95 = (
    (actual >= holdout_predictions["model_prediction"] * lower_95)
    & (actual <= holdout_predictions["model_prediction"] * upper_95)
).mean() * 100

print("FINAL HOLDOUT RESULTS")
print("=" * 45)
print(f"Holdout rows: {len(holdout_predictions):,}")
print(f"Common comparison rows: {len(comparison):,}")
print(f"Model WAPE (all rows): {model_wape_all:.3f}%")
print(f"Model WAPE (common rows): {model_common_wape:.3f}%")
print(f"Lag-364 baseline WAPE: {best_baseline_wape:.3f}%")
print(f"Improvement vs lag-364: {improvement_pp:.3f} percentage points")
print(f"Model forecast/actual ratio (all rows): {model_bias_all:.4f}")

print("\nSame-row comparison:")
for name, metrics in comparison_metrics.items():
    print(
        f"{name}: WAPE={metrics['wape_pct']:.3f}%, "
        f"forecast/actual={metrics['forecast_actual_ratio']:.4f}"
    )

print(f"\n90% band empirical coverage: {coverage_90:.2f}%")
print(f"95% band empirical coverage: {coverage_95:.2f}%")

# Save scored predictions for auditability
from pathlib import Path

output_dir = Path("../reports")
output_dir.mkdir(parents=True, exist_ok=True)

holdout_predictions.to_csv(
    output_dir / "holdout_predictions.csv",
    index=False
)

print("\nSaved: reports/holdout_predictions.csv")

FINAL HOLDOUT RESULTS
Holdout rows: 40,282
Common comparison rows: 32,603
Model WAPE (all rows): 9.326%
Model WAPE (common rows): 9.117%
Lag-364 baseline WAPE: 11.283%
Improvement vs lag-364: 2.166 percentage points
Model forecast/actual ratio (all rows): 1.0155

Same-row comparison:
model_prediction: WAPE=9.117%, forecast/actual=1.0128
pred_lag42: WAPE=14.026%, forecast/actual=1.0383
pred_lag364: WAPE=11.283%, forecast/actual=0.9764
pred_weekday_mean: WAPE=16.995%, forecast/actual=1.0531

90% band empirical coverage: 92.13%
95% band empirical coverage: 96.32%

Saved: reports/holdout_predictions.csv


In [ ]:
# Cell 43: Store-level bootstrap for holdout improvement

rng = np.random.default_rng(42)
n_bootstrap = 2000

# Use only rows where both methods have predictions
boot_data = comparison.copy()
boot_data["model_abs_error"] = (
    boot_data["Sales"] - boot_data["model_prediction"]
).abs()
boot_data["baseline_abs_error"] = (
    boot_data["Sales"] - boot_data["pred_lag364"]
).abs()

# Aggregate errors and actual sales by store
store_errors = (
    boot_data.groupby("Store")
    .agg(
        model_error=("model_abs_error", "sum"),
        baseline_error=("baseline_abs_error", "sum"),
        actual_sales=("Sales", "sum")
    )
)

store_ids = store_errors.index.to_numpy()
bootstrap_differences = []

for _ in range(n_bootstrap):
    sampled_stores = rng.choice(
        store_ids, size=len(store_ids), replace=True
    )
    sampled = store_errors.loc[sampled_stores]

    model_wape = (
        sampled["model_error"].sum()
        / sampled["actual_sales"].sum()
        * 100
    )
    baseline_wape = (
        sampled["baseline_error"].sum()
        / sampled["actual_sales"].sum()
        * 100
    )

    # Positive values mean the model has lower WAPE
    bootstrap_differences.append(baseline_wape - model_wape)

bootstrap_differences = np.array(bootstrap_differences)

ci_low, ci_high = np.percentile(bootstrap_differences, [2.5, 97.5])

print(f"Stores resampled: {len(store_ids)}")
print(f"Bootstrap repetitions: {n_bootstrap}")
print(f"Observed WAPE improvement: {improvement_pp:.3f} percentage points")
print(f"95% bootstrap interval: [{ci_low:.3f}, {ci_high:.3f}] percentage points")

if ci_low > 0:
    print("Interpretation: the interval supports a positive improvement.")
elif ci_high < 0:
    print("Interpretation: the interval supports the baseline performing better.")
else:
    print("Interpretation: the interval includes zero; improvement is uncertain.")

Stores resampled: 1115
Bootstrap repetitions: 2000
Observed WAPE improvement: 2.166 percentage points
95% bootstrap interval: [1.932, 2.402] percentage points
Interpretation: the interval supports a positive improvement.


In [ ]:
# Cell 44: Save final evaluation results to results.json

import json
from pathlib import Path
from datetime import datetime

# Validation results recorded from the four completed folds
validation_results = {
    "fold_wape_pct": [
        11.359712,
        8.984187,
        10.584053,
        8.826204
    ],
    "mean_wape_pct": 9.938539,
    "min_wape_pct": 8.826204,
    "max_wape_pct": 11.359712,
    "best_baseline": "Sales_lag_364",
    "best_baseline_mean_wape_pct": 14.654713,
    "improvement_vs_best_baseline_pp": 4.716174
}

# Final holdout results already calculated in Cell 42B
holdout_results = {
    "period_start": "2015-06-20",
    "period_end": "2015-07-31",
    "holdout_open_store_days": int(len(holdout_predictions)),
    "common_comparison_rows": int(len(comparison)),
    "model_wape_all_rows_pct": float(model_wape_all),
    "model_wape_common_rows_pct": float(model_common_wape),
    "lag_364_wape_common_rows_pct": float(
        comparison_metrics["pred_lag364"]["wape_pct"]
    ),
    "lag_42_wape_common_rows_pct": float(
        comparison_metrics["pred_lag42"]["wape_pct"]
    ),
    "weekday_mean_wape_common_rows_pct": float(
        comparison_metrics["pred_weekday_mean"]["wape_pct"]
    ),
    "improvement_vs_lag_364_pp": float(improvement_pp),
    "forecast_actual_ratio_all_rows": float(model_bias_all),
    "band_90_empirical_coverage_pct": float(coverage_90),
    "band_95_empirical_coverage_pct": float(coverage_95)
}

bootstrap_results = {
    "method": "Store-level bootstrap",
    "repetitions": int(n_bootstrap),
    "stores_resampled": int(len(store_ids)),
    "improvement_point_estimate_pp": float(improvement_pp),
    "ci_95_lower_pp": float(ci_low),
    "ci_95_upper_pp": float(ci_high),
    "interpretation": (
        "The store-level bootstrap interval is above zero; "
        "the model has lower WAPE than the lag-364 baseline "
        "on the evaluated holdout rows."
    )
}

results = {
    "project": "Retail Demand Forecasting",
    "selected_model": "HistGradientBoostingRegressor",
    "forecast_horizon_days": 42,
    "selection_rule": (
        "Choose the model with the lowest mean WAPE across four "
        "rolling validation folds; prefer the simpler baseline unless "
        "the improvement is at least 1 percentage point."
    ),
    "validation": validation_results,
    "holdout": holdout_results,
    "uncertainty": bootstrap_results,
    "prediction_band_multipliers": {
        "90_percent": {
            "lower": float(lower_90),
            "upper": float(upper_90)
        },
        "95_percent": {
            "lower": float(lower_95),
            "upper": float(upper_95)
        }
    },
    "caveats": [
        "The holdout covers only 42 calendar days in 2015.",
        "Bootstrap uncertainty is across stores conditional on this holdout period.",
        "Prediction-band coverage is empirical on this holdout, not a guarantee of future coverage.",
        "Baseline comparisons use only rows where all compared methods have predictions."
    ]
}

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

results_path = reports_dir / "results.json"

with results_path.open("w", encoding="utf-8") as file:
    json.dump(results, file, indent=2, allow_nan=False)

# Read it back to verify valid JSON
with results_path.open("r", encoding="utf-8") as file:
    verified_results = json.load(file)

print("Saved and verified:", results_path.resolve())
print("Model holdout WAPE:", verified_results["holdout"]["model_wape_all_rows_pct"])
print("Holdout improvement (pp):", verified_results["holdout"]["improvement_vs_lag_364_pp"])
print("Bootstrap 95% interval:",
      verified_results["uncertainty"]["ci_95_lower_pp"],
      "to",
      verified_results["uncertainty"]["ci_95_upper_pp"])

Saved and verified: C:\Users\KISHAN PANWAR\OneDrive\Documents\Desktop\DATASCIENCE\Projects\rossmann-demand-forecasting\reports\results.json
Model holdout WAPE: 9.32554244934457
Holdout improvement (pp): 2.165962304495256
Bootstrap 95% interval: 1.931989940173496 to 2.4022874350278354


In [ ]:
# Cell 45: Inspect project structure before building the pipeline

from pathlib import Path

cwd = Path.cwd().resolve()

print("Current notebook working directory:")
print(cwd)

# Find the project root from the current directory or its parents
project_root = next(
    (
        path for path in [cwd, *cwd.parents]
        if path.name == "rossmann-demand-forecasting"
    ),
    None
)

if project_root is None:
    print("\nProject root not found automatically.")
    print("Please share the working directory printed above.")
else:
    print("\nProject root:", project_root)
    print("\nProject files and folders:")

    for path in sorted(project_root.rglob("*")):
        if any(
            part in {".git", ".ipynb_checkpoints", "__pycache__", ".venv"}
            for part in path.parts
        ):
            continue

        if path.is_file():
            print(f"  FILE  {path.relative_to(project_root)}")
        elif path.is_dir():
            print(f"  DIR   {path.relative_to(project_root)}/")

Current notebook working directory:
C:\Users\KISHAN PANWAR\OneDrive\Documents\Desktop\DATASCIENCE\Projects\rossmann-demand-forecasting\notebooks

Project root: C:\Users\KISHAN PANWAR\OneDrive\Documents\Desktop\DATASCIENCE\Projects\rossmann-demand-forecasting

Project files and folders:
  DIR   data/
  FILE  data\rossmann.db
  FILE  data\store.csv
  FILE  data\test.csv
  FILE  data\train.csv
  FILE  DECISIONS.md
  DIR   notebooks/
  FILE  notebooks\01_data_audit.ipynb
  FILE  README.md
  DIR   reports/
  FILE  reports\holdout_predictions.csv
  FILE  reports\results.json


In [ ]:
# Inspect the notebook's model feature order
print("Number of features:", len(feature_columns))
for i, column in enumerate(feature_columns):
    print(i, column)

Number of features: 36
0 Promo
1 SchoolHoliday
2 DayOfWeek
3 Month
4 DayOfMonth
5 WeekOfYear
6 IsWeekend
7 Sales_lag_42
8 Sales_lag_49
9 Sales_lag_364
10 Sales_same_weekday_mean_8w
11 Sales_rolling_mean_56d
12 CompetitionDistance
13 CompetitionOpenSinceMonth
14 CompetitionOpenSinceYear
15 Promo2
16 Promo2SinceWeek
17 Promo2SinceYear
18 StateHoliday_0
19 StateHoliday_a
20 StateHoliday_b
21 StateHoliday_c
22 StateHoliday_<NA>
23 StoreType_a
24 StoreType_b
25 StoreType_c
26 StoreType_d
27 StoreType_<NA>
28 Assortment_a
29 Assortment_b
30 Assortment_c
31 Assortment_<NA>
32 PromoInterval_Feb,May,Aug,Nov
33 PromoInterval_Jan,Apr,Jul,Oct
34 PromoInterval_Mar,Jun,Sept,Dec
35 PromoInterval_<NA>


In [ ]:
# Compare missingness and feature values in the existing notebook table
sales_features = [
    "Sales_lag_42",
    "Sales_lag_49",
    "Sales_lag_364",
    "Sales_same_weekday_mean_8w",
    "Sales_rolling_mean_56d",
]

print("Notebook feature shape:", features.shape)
print("\nMissing counts:")
print(features[sales_features].isna().sum())

Notebook feature shape: (844392, 39)

Missing counts:
Sales_lag_42                   75867
Sales_lag_49                   79610
Sales_lag_364                 359515
Sales_same_weekday_mean_8w     46866
Sales_rolling_mean_56d         46428
dtype: int64


In [ ]:
# Compare exact feature values on a deterministic sample
check_columns = [
    "Store", "Date", "Sales",
    "Sales_lag_42",
    "Sales_lag_49",
    "Sales_lag_364",
    "Sales_same_weekday_mean_8w",
    "Sales_rolling_mean_56d",
]

notebook_check = (
    features[check_columns]
    .sort_values(["Store", "Date"])
    .iloc[::1000]
    .reset_index(drop=True)
)

print(notebook_check.head(10).to_string(index=False))

 Store       Date  Sales  Sales_lag_42  Sales_lag_49  Sales_lag_364  Sales_same_weekday_mean_8w  Sales_rolling_mean_56d
     1 2013-01-02   5530           NaN           NaN            NaN                         NaN                     NaN
     2 2013-09-20   3949        3551.0        4776.0            NaN                 4642.500000             4829.187500
     3 2014-06-12   6889           NaN        5389.0         5499.0                 6305.000000             6554.888889
     4 2015-02-27  10243        9008.0        8658.0        10849.0                10865.714286            11227.066667
     6 2013-04-23   7543        5579.0        8057.0            NaN                 6310.875000             5884.062500
     7 2014-01-11   4971        8442.0        5610.0         4236.0                 5558.000000             8135.062500
     8 2014-09-25   5233        4588.0        6240.0         6475.0                 5868.000000             5188.937500
     9 2015-06-20   6179        7347.0  

In [ ]:
# Inspect columns before comparing
print(fold_predictions.columns.tolist())

AttributeError: 'list' object has no attribute 'columns'

In [ ]:
# Inspect the structure of fold_predictions
print("Type:", type(fold_predictions))
print("Number of items:", len(fold_predictions))

if fold_predictions:
    print("First item type:", type(fold_predictions[0]))
    print("First item:", fold_predictions[0])
    if isinstance(fold_predictions[0], dict):
        print("First item keys:", fold_predictions[0].keys())
    elif hasattr(fold_predictions[0], "columns"):
        print("First item columns:", fold_predictions[0].columns.tolist())

Type: <class 'list'>
Number of items: 4
First item type: <class 'pandas.DataFrame'>
First item:          Store       Date  Sales  pred_lag42  pred_lag364  pred_weekday_mean  \
732          1 2015-01-03   5023      5849.0       5173.0        5116.375000   
734          1 2015-01-05   6239      6527.0       6194.0        4973.375000   
735          1 2015-01-06   4574      5815.0       5539.0        4574.875000   
736          1 2015-01-07   4796      5328.0       4931.0        4510.625000   
737          1 2015-01-08   3716      6144.0       4396.0        4544.125000   
...        ...        ...    ...         ...          ...                ...   
1017034   1115 2015-02-07   7218      6990.0       6575.0        8219.500000   
1017036   1115 2015-02-09   5604      7167.0       4283.0        8559.625000   
1017037   1115 2015-02-10   4714      7148.0       3937.0        7916.125000   
1017038   1115 2015-02-11   5206      3847.0       3982.0        6787.250000   
1017040   1115 2015-02-1

In [ ]:
# Inspect each fold's columns and the objects holding model predictions
for i, fold_df in enumerate(fold_predictions, start=1):
    print(f"Fold {i}: {fold_df.shape}, columns={fold_df.columns.tolist()}")

print("\nModel prediction variables:")
for name in ["validation_rows", "model_predictions", "oof_predictions"]:
    if name in globals():
        obj = globals()[name]
        print(name, type(obj), getattr(obj, "shape", None))
        if hasattr(obj, "columns"):
            print("  Columns:", obj.columns.tolist())

Fold 1: (30832, 7), columns=['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'fold']
Fold 2: (39703, 7), columns=['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'fold']
Fold 3: (33600, 7), columns=['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'fold']
Fold 4: (31279, 7), columns=['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'fold']

Model prediction variables:
validation_rows <class 'pandas.DataFrame'> (37473, 40)
  Columns: ['Store', 'Date', 'Sales', 'Promo', 'SchoolHoliday', 'DayOfWeek', 'Month', 'DayOfMonth', 'WeekOfYear', 'IsWeekend', 'Sales_lag_42', 'Sales_lag_49', 'Sales_lag_364', 'Sales_same_weekday_mean_8w', 'Sales_rolling_mean_56d', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'StateHoliday_0', 'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c', 'StateHoliday_nan', 'StoreType_a

In [ ]:
# Compare model and baseline WAPE on identical rows
for fold_num, baseline_df in enumerate(fold_predictions, start=1):
    start_date = baseline_df["Date"].min()
    end_date = baseline_df["Date"].max()

    model_df = validation_rows[
        (validation_rows["Date"] >= start_date)
        & (validation_rows["Date"] <= end_date)
    ].copy()

    # Restrict model predictions to the exact Store-Date pairs in this fold
    common = baseline_df[["Store", "Date", "Sales"]].merge(
        model_df[["Store", "Date", "Sales", "model_prediction"]],
        on=["Store", "Date", "Sales"],
        how="inner",
        validate="one_to_one"
    )

    wape = (
        100 * (common["Sales"] - common["model_prediction"]).abs().sum()
        / common["Sales"].abs().sum()
    )

    print(
        f"Fold {fold_num}: "
        f"baseline rows={len(baseline_df):,}, "
        f"matched rows={len(common):,}, "
        f"model WAPE={wape:.6f}%"
    )

Fold 1: baseline rows=30,832, matched rows=0, model WAPE=nan%
Fold 2: baseline rows=39,703, matched rows=0, model WAPE=nan%
Fold 3: baseline rows=33,600, matched rows=0, model WAPE=nan%
Fold 4: baseline rows=31,279, matched rows=31,279, model WAPE=8.826204%


C:\Users\KISHAN PANWAR\AppData\Local\Temp\ipykernel_14052\1437826170.py:20: RuntimeWarning: invalid value encountered in scalar divide
  100 * (common["Sales"] - common["model_prediction"]).abs().sum()
C:\Users\KISHAN PANWAR\AppData\Local\Temp\ipykernel_14052\1437826170.py:20: RuntimeWarning: invalid value encountered in scalar divide
  100 * (common["Sales"] - common["model_prediction"]).abs().sum()
C:\Users\KISHAN PANWAR\AppData\Local\Temp\ipykernel_14052\1437826170.py:20: RuntimeWarning: invalid value encountered in scalar divide
  100 * (common["Sales"] - common["model_prediction"]).abs().sum()


In [ ]:
# Diagnose fold date ranges and Store-Date key overlap
for fold_num, baseline_df in enumerate(fold_predictions, start=1):
    start_date = baseline_df["Date"].min()
    end_date = baseline_df["Date"].max()

    model_df = validation_rows[
        validation_rows["Date"].between(start_date, end_date)
    ]

    baseline_keys = baseline_df[["Store", "Date"]].drop_duplicates()
    model_keys = model_df[["Store", "Date"]].drop_duplicates()

    overlap = baseline_keys.merge(
        model_keys,
        on=["Store", "Date"],
        how="inner"
    )

    print(f"\nFold {fold_num}")
    print("Baseline dates:", start_date, "to", end_date)
    print("Model rows in date range:", len(model_df))
    print("Baseline unique keys:", len(baseline_keys))
    print("Model unique keys:", len(model_keys))
    print("Matching Store-Date keys:", len(overlap))


Fold 1
Baseline dates: 2015-01-03 00:00:00 to 2015-02-13 00:00:00
Model rows in date range: 0
Baseline unique keys: 30832
Model unique keys: 0
Matching Store-Date keys: 0

Fold 2
Baseline dates: 2015-02-14 00:00:00 to 2015-03-27 00:00:00
Model rows in date range: 0
Baseline unique keys: 39703
Model unique keys: 0
Matching Store-Date keys: 0

Fold 3
Baseline dates: 2015-03-28 00:00:00 to 2015-05-08 00:00:00
Model rows in date range: 0
Baseline unique keys: 33600
Model unique keys: 0
Matching Store-Date keys: 0

Fold 4
Baseline dates: 2015-05-09 00:00:00 to 2015-06-19 00:00:00
Model rows in date range: 37473
Baseline unique keys: 31279
Model unique keys: 37473
Matching Store-Date keys: 31279


In [ ]:
# Inspect notebook variables that may retain out-of-fold predictions
for name in [
    "fold_predictions",
    "validation_rows",
    "oof_predictions",
    "all_validation_predictions",
    "model_validation_predictions",
]:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}: {type(obj).__name__}")
        if isinstance(obj, list):
            print("  Items:", len(obj))
            for i, item in enumerate(obj, start=1):
                if hasattr(item, "columns"):
                    print(
                        f"  Item {i}: rows={len(item):,}, "
                        f"columns={item.columns.tolist()}"
                    )
        elif hasattr(obj, "shape"):
            print("  Shape:", obj.shape)
            if hasattr(obj, "columns"):
                print("  Columns:", obj.columns.tolist())
                if "Date" in obj.columns:
                    print("  Date range:", obj["Date"].min(), "to", obj["Date"].max())


fold_predictions: list
  Items: 4
  Item 1: rows=30,832, columns=['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'fold']
  Item 2: rows=39,703, columns=['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'fold']
  Item 3: rows=33,600, columns=['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'fold']
  Item 4: rows=31,279, columns=['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'fold']

validation_rows: DataFrame
  Shape: (37473, 40)
  Columns: ['Store', 'Date', 'Sales', 'Promo', 'SchoolHoliday', 'DayOfWeek', 'Month', 'DayOfMonth', 'WeekOfYear', 'IsWeekend', 'Sales_lag_42', 'Sales_lag_49', 'Sales_lag_364', 'Sales_same_weekday_mean_8w', 'Sales_rolling_mean_56d', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'StateHoliday_0', 'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c', 'StateHoliday_nan', 

In [ ]:
# Inspect saved out-of-fold model and baseline predictions
print("Saved fold prediction objects:", len(model_fold_predictions))

for fold_id, df in enumerate(model_fold_predictions, start=1):
    print(
        f"Fold {fold_id}: rows={len(df):,}, "
        f"columns={df.columns.tolist()}"
    )

print("\nSaved model WAPE by fold:")
saved_metrics = all_fold_metrics[
    all_fold_metrics["method"] == "HistGradientBoostingRegressor"
][["fold", "WAPE_pct", "eligible_targets"]]

print(saved_metrics.to_string(index=False))

Saved fold prediction objects: 4
Fold 1: rows=30,832, columns=['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'model_prediction', 'fold']
Fold 2: rows=39,703, columns=['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'model_prediction', 'fold']
Fold 3: rows=33,600, columns=['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'model_prediction', 'fold']
Fold 4: rows=31,279, columns=['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'model_prediction', 'fold']

Saved model WAPE by fold:
 fold  WAPE_pct  eligible_targets
    1 11.359712             30832
    2  8.984187             39703
    3 10.584053             33600
    4  8.826204             31279


In [ ]:
# Compare notebook WAPE with the pipeline's row-selection logic
baseline_columns = [
    "pred_lag42",
    "pred_lag364",
    "pred_weekday_mean",
]

for fold_id, notebook_df in enumerate(model_fold_predictions, start=1):
    common = notebook_df.dropna(
        subset=["Sales", "model_prediction"] + baseline_columns
    )

    notebook_wape = (
        100
        * (common["Sales"] - common["model_prediction"]).abs().sum()
        / common["Sales"].abs().sum()
    )

    print(
        f"Fold {fold_id}: "
        f"rows={len(common):,}, "
        f"WAPE={notebook_wape:.6f}%"
    )

Fold 1: rows=30,832, WAPE=11.359712%
Fold 2: rows=39,703, WAPE=8.984187%
Fold 3: rows=33,600, WAPE=10.584053%
Fold 4: rows=31,279, WAPE=8.826204%


In [ ]:
# Compare notebook and pipeline predictions row by row
import pandas as pd
notebook_oof = pd.read_csv("notebook_validation_predictions.csv", parse_dates=["Date"])

pipeline_oof = pd.read_csv(
    "../reports/pipeline_validation_predictions.csv",
    parse_dates=["Date"]
)

comparison = notebook_oof.merge(
    pipeline_oof,
    on=["Store", "Date", "Sales", "fold"],
    how="inner",
    suffixes=("_notebook", "_pipeline"),
    validate="one_to_one"
)

print("Notebook rows:", len(notebook_oof))
print("Pipeline rows:", len(pipeline_oof))
print("Matched rows:", len(comparison))

comparison["absolute_prediction_difference"] = (
    comparison["model_prediction_notebook"]
    - comparison["model_prediction_pipeline"]
).abs()

print(
    "\nPrediction difference summary:"
)
print(comparison["absolute_prediction_difference"].describe())

print("\nLargest prediction differences:")
print(
    comparison.sort_values(
        "absolute_prediction_difference",
        ascending=False
    )[
        [
            "Store", "Date", "fold", "Sales",
            "model_prediction_notebook",
            "model_prediction_pipeline",
            "absolute_prediction_difference"
        ]
    ].head(10).to_string(index=False)
)

Notebook rows: 135414
Pipeline rows: 135414
Matched rows: 135414

Prediction difference summary:
count    135414.000000
mean        122.456081
std         123.562849
min           0.000246
25%          40.044042
50%          88.132291
75%         162.973763
max        2605.128758
Name: absolute_prediction_difference, dtype: float64

Largest prediction differences:
 Store       Date  fold  Sales  model_prediction_notebook  model_prediction_pipeline  absolute_prediction_difference
    85 2015-05-25     4  13018                6905.901617                9511.030375                     2605.128758
   262 2015-04-06     3  33655               20507.178849               22670.165104                     2162.986254
   562 2015-05-25     4  21370               16239.594489               18392.820813                     2153.226324
   262 2015-04-03     3  38722               25083.580735               27214.346353                     2130.765618
  1114 2015-04-23     3  17467               181

In [34]:
# Fingerprint notebook training data for each validation fold
import hashlib
import pandas as pd

# Rebuild the predictor list using the project's existing helper

# Derive predictor columns directly from the existing feature table
excluded = {"Sales", "Customers", "Date", "Open", "Store"}
feature_columns = [
    col for col in features.columns
    if col not in excluded
    and pd.api.types.is_numeric_dtype(features[col])
]

 

def dataframe_fingerprint(df):
    normalized = df.reset_index(drop=True).astype("float64")
    row_hashes = pd.util.hash_pandas_object(
        normalized, index=False
    ).to_numpy()
    return hashlib.sha256(row_hashes.tobytes()).hexdigest()[:16]

fold_ranges = [
    (1, "2015-01-03", "2015-02-13"),
    (2, "2015-02-14", "2015-03-27"),
    (3, "2015-03-28", "2015-05-08"),
    (4, "2015-05-09", "2015-06-19"),
]

for fold_id, start, end in fold_ranges:
    forecast_origin = pd.Timestamp(start) - pd.Timedelta(days=1)
    train_rows = features.loc[
        features["Date"] <= forecast_origin
    ].copy()

    print(
        f"Fold {fold_id}: rows={len(train_rows):,}, "
        f"X={dataframe_fingerprint(train_rows[feature_columns])}, "
        f"y={dataframe_fingerprint(train_rows[['Sales']])}"
    )

Fold 1: rows=649,493, X=6338f72460582947, y=f751c4f0683d3a89
Fold 2: rows=689,417, X=980192ba61353cd0, y=1502efca5e34a7c1
Fold 3: rows=729,593, X=e43fa3dccf26e9cb, y=9336a142d1f61c1f
Fold 4: rows=766,637, X=69dc7809a7f89647, y=aa6d7d4732a2799f


In [36]:
# Canonical training-data fingerprints
import hashlib
import pandas as pd

excluded = {"Sales", "Customers", "Date", "Open", "Store"}
# Use a stable, alphabetical predictor order for diagnostics
feature_columns = sorted(
    c for c in features.columns
    if c not in {"Sales", "Customers", "Date", "Open", "Store"}
    and pd.api.types.is_numeric_dtype(features[c])
)

def dataframe_fingerprint(df):
    normalized = df.reset_index(drop=True).astype("float64")
    hashes = pd.util.hash_pandas_object(
        normalized, index=False
    ).to_numpy()
    return hashlib.sha256(hashes.tobytes()).hexdigest()[:16]

fold_ranges = [
    (1, "2015-01-03", "2015-02-13"),
    (2, "2015-02-14", "2015-03-27"),
    (3, "2015-03-28", "2015-05-08"),
    (4, "2015-05-09", "2015-06-19"),
]

for fold_id, start, end in fold_ranges:
    origin = pd.Timestamp(start) - pd.Timedelta(days=1)
    train_rows = features.loc[features["Date"] <= origin].copy()
    train_rows = train_rows.sort_values(["Store", "Date"])

    X = train_rows[feature_columns]
    y = train_rows[["Sales"]]

    print(
        f"Fold {fold_id}: rows={len(train_rows):,}, "
        f"X={dataframe_fingerprint(X)}, "
        f"y={dataframe_fingerprint(y)}"
    )

Fold 1: rows=649,493, X=38a59b3e8cb3066a, y=f751c4f0683d3a89
Fold 2: rows=689,417, X=ffb1a2aaaa4e3053, y=1502efca5e34a7c1
Fold 3: rows=729,593, X=ec23662049fa54b4, y=9336a142d1f61c1f
Fold 4: rows=766,637, X=b2cb07a8ea348c47, y=aa6d7d4732a2799f


In [37]:
# Compare feature columns and values for the same Store-Date rows
excluded = {"Sales", "Customers", "Date", "Open", "Store"}

notebook_cols = sorted(
    c for c in features.columns
    if c not in excluded
    and pd.api.types.is_numeric_dtype(features[c])
)

print("Notebook predictor count:", len(notebook_cols))
print("First 10 predictors:", notebook_cols[:10])

# Confirm the pipeline prediction file has the expected validation rows
pipeline_oof = pd.read_csv(
    "../reports/pipeline_validation_predictions.csv",
    parse_dates=["Date"]
)

sample_keys = pipeline_oof[["Store", "Date"]].drop_duplicates().head(10)

sample_features = sample_keys.merge(
    features[["Store", "Date"] + notebook_cols],
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

print("Matched feature rows:", len(sample_features))
print(sample_features[["Store", "Date"] + notebook_cols[:5]].to_string(index=False))

Notebook predictor count: 10
First 10 predictors: ['DayOfMonth', 'DayOfWeek', 'IsWeekend', 'Month', 'Promo', 'Sales_lag_364', 'Sales_lag_42', 'Sales_lag_49', 'SchoolHoliday', 'WeekOfYear']
Matched feature rows: 10
 Store       Date  DayOfMonth  DayOfWeek  IsWeekend  Month  Promo
     1 2015-01-03           3          6          1      1      0
     1 2015-01-05           5          1          0      1      1
     1 2015-01-06           6          2          0      1      1
     1 2015-01-07           7          3          0      1      1
     1 2015-01-08           8          4          0      1      1
     1 2015-01-09           9          5          0      1      1
     1 2015-01-10          10          6          1      1      0
     1 2015-01-12          12          1          0      1      1
     1 2015-01-13          13          2          0      1      1
     1 2015-01-14          14          3          0      1      1


In [38]:
# Inspect all feature columns and their data types
print("Feature table shape:", features.shape)
print("\nAll columns and data types:")
print(features.dtypes.to_string())

print("\nPipeline feature count from project specification: 36")

Feature table shape: (844392, 14)

All columns and data types:
Store                     int64
Date             datetime64[us]
Sales                     int64
Promo                     int64
StateHoliday                str
SchoolHoliday             int64
DayOfWeek                 int32
Month                     int32
DayOfMonth                int32
WeekOfYear                int32
IsWeekend                 int32
Sales_lag_42            float64
Sales_lag_49            float64
Sales_lag_364           float64

Pipeline feature count from project specification: 36


In [39]:
# Inspect the saved full-model feature order from Cell 35
print("Notebook feature_columns exists:", "feature_columns" in globals())

if "feature_columns" in globals():
    print("Notebook feature count:", len(feature_columns))
    print("\nNotebook feature order:")
    print(feature_columns)

Notebook feature_columns exists: True
Notebook feature count: 10

Notebook feature order:
['DayOfMonth', 'DayOfWeek', 'IsWeekend', 'Month', 'Promo', 'Sales_lag_364', 'Sales_lag_42', 'Sales_lag_49', 'SchoolHoliday', 'WeekOfYear']


In [41]:
# Inspect the saved notebook predictions and available model-related variables

print("Model-related variables:")
print([
    name for name in globals()
    if any(word in name.lower() for word in
           ["model", "predict", "feature_columns", "fold"])
])

print("\nSaved notebook predictions:")
print("model_fold_predictions exists:",
      "model_fold_predictions" in globals())

if "model_fold_predictions" in globals():
    print("Number of folds:", len(model_fold_predictions))

    for i, df in enumerate(model_fold_predictions, start=1):
        print(f"\nFold {i}")
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(2))

Model-related variables:
['folds', 'fold_number', 'fold_end', 'fold_start', 'fold_table', 'fold', 'predict_baselines', 'first_fold', 'fold_targets', 'baseline_predictions', 'prediction_columns', 'fold_results', 'fold_predictions', 'predictions', 'fold_ranges', 'fold_id', 'feature_columns']

Saved notebook predictions:
model_fold_predictions exists: False


In [42]:
# Inspect the remaining saved fold predictions

print("fold_predictions type:", type(fold_predictions))
print("Number of folds:", len(fold_predictions))

for i, df in enumerate(fold_predictions, start=1):
    print(f"\nFold {i}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(2))

fold_predictions type: <class 'list'>
Number of folds: 4

Fold 1
Shape: (30832, 7)
Columns: ['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'fold']
     Store       Date  Sales  pred_lag42  pred_lag364  pred_weekday_mean  fold
732      1 2015-01-03   5023      5849.0       5173.0           5116.375     1
734      1 2015-01-05   6239      6527.0       6194.0           4973.375     1

Fold 2
Shape: (39703, 7)
Columns: ['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'fold']
     Store       Date  Sales  pred_lag42  pred_lag364  pred_weekday_mean  fold
774      1 2015-02-14   5015      5023.0       5258.0           6347.125     2
776      1 2015-02-16   4303      6239.0       5393.0           6559.375     2

Fold 3
Shape: (33600, 7)
Columns: ['Store', 'Date', 'Sales', 'pred_lag42', 'pred_lag364', 'pred_weekday_mean', 'fold']
     Store       Date  Sales  pred_lag42  pred_lag364  pred_weekday_mean  fold
816      1 2015-03-28   5208  

In [43]:
# Extract Cell 35 directly from the saved notebook file

import json
from pathlib import Path

notebook_path = Path(
    r"C:\Users\KISHAN PANWAR\OneDrive\Documents\Desktop"
    r"\DATASCIENCE\Projects\rossmann-demand-forecasting"
    r"\notebooks\01_data_audit.ipynb"
)

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook["cells"]

print("Total cells:", len(cells))
print("Cell 35 type:", cells[34]["cell_type"])
print("\n--- Cell 35 source ---\n")
print("".join(cells[34]["source"]))

Total cells: 76
Cell 35 type: code

--- Cell 35 source ---

# Cell 33: Recheck feature provenance after repairing rolling alignment

rng = np.random.default_rng(42)
sample_indices = rng.choice(
    features.index.to_numpy(),
    size=min(300, len(features)),
    replace=False
)
sample = features.loc[sample_indices].copy()

history_by_store = {
    store_id: group.sort_values("Date")
    for store_id, group in feature_data.groupby("Store")
}

for _, row in sample.iterrows():
    store_id = row["Store"]
    target_date = pd.Timestamp(row["Date"])
    cutoff = target_date - pd.Timedelta(days=42)

    # Exact lags
    for lag in [42, 49, 364]:
        expected = expected_sales(
            store_id, target_date - pd.Timedelta(days=lag)
        )
        actual = row[f"Sales_lag_{lag}"]

        assert (
            (pd.isna(expected) and pd.isna(actual))
            or (
                pd.notna(expected)
                and pd.notna(actual)
                and np.isclose(expected, actual)

In [44]:
# Find notebook cells containing model training and validation predictions

for i, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))

    if any(term in source for term in [
        "HistGradientBoostingRegressor",
        "model_fold_predictions",
        "model_prediction"
    ]):
        print(f"\n{'=' * 60}")
        print(f"Cell index: {i} | Type: {cell['cell_type']}")
        print(source[:5000])


Cell index: 37 | Type: code
# Cell 36: Analyze validation errors by business segment

validation_predictions = pd.concat(
    model_fold_predictions,
    ignore_index=True
)

# Attach known target-day attributes and store type for analysis.
analysis_columns = features[
    ["Store", "Date", "Promo", "SchoolHoliday"]
].drop_duplicates(["Store", "Date"])

validation_predictions = validation_predictions.merge(
    analysis_columns,
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

store_types = store_metadata[["Store", "StoreType"]].drop_duplicates("Store")

validation_predictions = validation_predictions.merge(
    store_types,
    on="Store",
    how="left",
    validate="many_to_one"
)

validation_predictions["abs_error"] = (
    validation_predictions["Sales"]
    - validation_predictions["model_prediction"]
).abs()

validation_predictions["signed_error"] = (
    validation_predictions["model_prediction"]
    - validation_predictions["Sales"]
)

def summarize_seg

In [46]:
# Locate the original rolling-validation training loop

for i, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))

    if (
        "model_fold_predictions" in source
        and "model.fit(" in source
        and "for fold" in source
    ):
        print(f"\n{'=' * 60}")
        print(f"Cell index: {i}")
        print(source)

In [47]:
# Find every cell mentioning HistGradientBoosting or model.fit

for i, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))
    outputs = cell.get("outputs", [])

    if (
        "HistGradientBoosting" in source
        or ".fit(" in source
        or "all_fold_metrics" in source
    ):
        print(
            f"Cell index: {i} | "
            f"Execution count: {cell.get('execution_count')} | "
            f"Outputs: {len(outputs)}"
        )
        print(source[:500].replace("\n", " "))
        print("-" * 60)

Cell index: 35 | Execution count: None | Outputs: 2
# Cell 34: Define and audit allowed model features  excluded_columns = {     "Sales",     "Customers",     "Date",     "Open",     "Store"  # Keep store identity out initially; use static store attributes instead. }  feature_columns = [     column for column in features.columns     if column not in excluded_columns ]  forbidden_predictors = {"Sales", "Customers", "Open", "Date"} assert not (set(feature_columns) & forbidden_predictors)  # All predictors must be numeric for this initial HistGradie
------------------------------------------------------------
Cell index: 42 | Execution count: None | Outputs: 1
# Cell 41: One-time final holdout evaluation  from sklearn.ensemble import HistGradientBoostingRegressor  # Fit only on labels available before the holdout starts. final_train_rows = features.loc[     features["Date"] < holdout_start ].copy()  final_holdout_rows = features.loc[     features["Date"].between(holdout_start, holdout_end

In [48]:
# Inspect the saved feature-audit cell and its outputs

cell = notebook["cells"][35]

print("=== CELL SOURCE ===")
print("".join(cell["source"]))

print("\n=== SAVED OUTPUTS ===")
for output in cell.get("outputs", []):
    if "text" in output:
        print("".join(output["text"]))
    elif "data" in output and "text/plain" in output["data"]:
        print("".join(output["data"]["text/plain"]))

=== CELL SOURCE ===
# Cell 34: Define and audit allowed model features

excluded_columns = {
    "Sales",
    "Customers",
    "Date",
    "Open",
    "Store"  # Keep store identity out initially; use static store attributes instead.
}

feature_columns = [
    column for column in features.columns
    if column not in excluded_columns
]

forbidden_predictors = {"Sales", "Customers", "Open", "Date"}
assert not (set(feature_columns) & forbidden_predictors)

# All predictors must be numeric for this initial HistGradientBoosting model.
non_numeric = features[feature_columns].select_dtypes(
    exclude=[np.number, "bool"]
).columns.tolist()

print("Number of candidate predictors:", len(feature_columns))
print("Non-numeric predictors:", non_numeric)
print("Target column: Sales")
print("Customers included:", "Customers" in feature_columns)
print("Store ID included:", "Store" in feature_columns)

display(pd.DataFrame({"feature": feature_columns}))

=== SAVED OUTPUTS ===
Number of candidate pre

In [49]:
# Find cells that train the model on the audited feature columns

for i, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))

    if (
        "HistGradientBoostingRegressor" in source
        and "feature_columns" in source
        and ".fit(" in source
        and "holdout" not in source.lower()
    ):
        print(f"\n{'=' * 60}")
        print(f"Cell index: {i}")
        print(source)

In [50]:
# Find validation cells using model fitting, fold metrics, or saved predictions

terms = [
    "HistGradientBoostingRegressor",
    "model_fold_predictions",
    "all_fold_metrics",
    "Fold 1",
    "WAPE_pct",
]

for i, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))

    if any(term in source for term in terms):
        print(f"\n{'=' * 60}")
        print(f"Cell index: {i}")
        print(source[:2500])

        for output in cell.get("outputs", []):
            text_output = output.get("text", [])
            if text_output:
                print("\nSaved output:")
                print("".join(text_output)[:1500])


Cell index: 25

# Cell 24: Evaluate Baselines on Common Eligible Targets Across Four Folds

baseline_columns = [
    "pred_lag42",
    "pred_lag364",
    "pred_weekday_mean"
]

fold_results = []
fold_predictions = []

for fold in folds:
    targets = sales_data[
        sales_data["Date"].between(
            fold["validation_start"],
            fold["validation_end"]
        )
    ].copy()

    predictions = predict_baselines(
        history=sales_data,
        targets=targets
    )

    # Use the intersection of eligible targets for a fair comparison
    common = predictions.dropna(subset=baseline_columns).copy()

    print(
        f"Fold {fold['fold']}: {len(targets):,} open targets; "
        f"{len(common):,} common eligible targets"
    )

    actual = common["Sales"].to_numpy(dtype=float)
    total_actual = actual.sum()

    if total_actual <= 0:
        raise ValueError("WAPE requires positive total actual sales.")

    for column in baseline_columns:
        forecast = com